# Procesamiento de Imágenes MRI Fetales — Diagnóstico de Ventriculomegalia
## Pipeline Unificado (Clásico v3 + CNN Transfer Learning) · Versión Final Comentada
### Tecnológico de Monterrey · Ingeniería Biomédica · Equipo 3 · Junio 2026

> **Integrantes:** Ana Sofía López Romero · Andrea Nathaly Ochoa Torres · Viviana Alejandre Paniagua · Laisha Nicole Cortés Montés

---

## Criterio clínico — Cardoza et al. (1988)

| Clasificación   | Diámetro atrial |
|-----------------|-----------------|
| **Normal**      | < 10 mm         |
| **VM Leve**     | 10 – 12 mm      |
| **VM Moderada** | 12 – 15 mm      |
| **VM Severa**   | > 15 mm         |

El diámetro atrial se mide en el **plano axial transventricular**. El LCR aparece
**hiperintenso** en secuencias T2 (alto contenido de agua, T2 largo), lo que permite
segmentarlo por **umbralización basada en intensidad** (Prayer 2017; Glenn 2010).

---

## Estructura del notebook (alineada al objetivo SMART de la presentación)

| Sección | Etapa del reto | Objetivo SMART |
|---------|----------------|----------------|
| A — Metadatos e inventario | Base de datos | **S** Interfaz de carga/validación |
| B — Explorador interactivo | Selección de corte | **M** Corte transventricular |
| C — Etapa I: Análisis digital | Etapa I | matriz, resolución, bits, histograma |
| D — Funciones DSP del profesor | (base) | convolución, morfología, FFT manuales |
| E — Etapa II: Filtrado | Etapa II | reducción de ruido (Riciano) |
| F — Etapa III: Segmentación | Etapa III | **A** localizar ventrículos |
| G — Etapa IV-V: Cuantificación | Etapas IV-V | **R** diámetro atrial + clasificación |
| H — CNN Transfer Learning | Validación | **T** Attention U-Net + EfficientNet-B0 |
| I — Evaluación de métricas | Validación | Dice, IoU, Sens, Espec, Acc |
| J — Batch completo | Entrega | 10 volúmenes, mini-reporte 8 paneles |
| L — Análisis prioritario | casos clave | t2-t25 (JC axial) y 05-badrecon |
| M — Resúmenes presentación | diapositivas | figuras listas para slides |
| **Q&A — Defensa** | **(preguntas del jurado)** | **por qué CNN, Dice vs IoU, kernels, U-Net…** |

> **IMPORTANTE — orden de ejecución:** ejecutar las celdas **en orden**. Las Secciones C, E, F, G, H
> usan las funciones definidas en la **Sección D**. La capa de orientación (ventrículos hacia abajo)
> y la segmentación bilateral (L=azul / R=rojo) también viven en la Sección D.

> **Lee la Sección Q&A al final**: contiene la justificación técnica de cada decisión
> (por qué Gaussiano y no mediana, por qué este kernel, por qué CNN y no otro clasificador,
> qué es Dice vs IoU, por qué Attention U-Net + EfficientNet-B0, etc.).


In [ ]:
# ============================================================
# INSTALACIÓN DE DEPENDENCIAS — GOOGLE COLAB
# Ejecutar esta celda primero (solo tarda la primera vez).
# Colab ya incluye: torch, torchvision, opencv, scikit-image,
# scipy, matplotlib, pandas e ipywidgets.
# Solo faltan las siguientes librerias:
# ============================================================
!pip install -q nibabel segmentation-models-pytorch albumentations timm
print('OK  Dependencias instaladas.')


In [ ]:
import os
import shutil
from google.colab import drive

mount_path = "/content/drive"

if os.path.ismount(mount_path):
  print("Google Drive is already mounted.")
else:
  # Check if the mount_path directory exists and is not empty.
  # If it exists and contains files/subdirectories, it will cause the error.
  if os.path.exists(mount_path) and len(os.listdir(mount_path)) > 0:
    print(f"Warning: Mount point '{mount_path}' is not empty but not mounted. Clearing its contents.")
    try:
      # Attempt to remove all contents within /content/drive
      for item in os.listdir(mount_path):
        item_path = os.path.join(mount_path, item)
        if os.path.isfile(item_path) or os.path.islink(item_path):
          os.remove(item_path)
        elif os.path.isdir(item_path):
          shutil.rmtree(item_path)
      print(f"Successfully cleared contents of '{mount_path}'.")
    except Exception as e:
      print(f"Error clearing mount point '{mount_path}': {e}. Please restart runtime if issues persist.")

  print('Mounting Google Drive...')
  try:
    drive.mount(mount_path)
    print('OK  Google Drive mounted in /content/drive')
  except Exception as e:
    print(f'Error during mount: {e}')
    print('Please manually unmount or restart the runtime if issues persist.')

In [ ]:
# Importaciones — todas las librerias del pipeline en una sola celda
import os, warnings, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import nibabel as nib
import cv2
from scipy.ndimage import gaussian_filter, label as scipy_label, binary_fill_holes, binary_erosion
from skimage import exposure, measure
from skimage.filters import threshold_otsu
from skimage.morphology import disk, binary_closing, binary_opening, remove_small_objects
from skimage.transform import rotate
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    import segmentation_models_pytorch as smp
    SMP_DISPONIBLE = True
    print('OK  segmentation_models_pytorch disponible')
except ImportError:
    SMP_DISPONIBLE = False
    print('AVISO: smp no instalado — Seccion H desactivada')

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    ALB_DISPONIBLE = True
    print('OK  albumentations disponible')
except ImportError:
    ALB_DISPONIBLE = False

import ipywidgets as widgets
from IPython.display import display, clear_output

np.random.seed(42)
torch.manual_seed(42)
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100
print('OK  Importaciones completadas.')

---
## Sección A — Metadatos e Inventario de Datos
Carga todos los volúmenes definidos en CONFIG, imprime metadatos y genera mosaico.

In [ ]:
import os

BASE_DIR = '/content/drive/MyDrive/Ejemplos mri'

CONFIG = {
    'archivos': [
        # (ruta, tipo, plano)
        # hospital -> sagittal (data[x,:,:]); JC -> axial (data[:,:,z])
        (os.path.join(BASE_DIR, 'good quality',         '5110156', '56-good.nii'),    'hospital_good', 'sagittal'),
        (os.path.join(BASE_DIR, 'placenta bad quality', '5187149', '49-badrecon.nii'),'hospital_bad',  'sagittal'),
        (os.path.join(BASE_DIR, 'placenta bad quality', '5143066', '66-badrecon.nii'),'hospital_bad',  'sagittal'),
        (os.path.join(BASE_DIR, 'placenta bad quality', '5188605', '05-badrecon.nii'),'hospital_bad',  'sagittal'),
        (os.path.join(BASE_DIR, 'placenta bad quality', '5189896', '96-badrecon.nii'),'hospital_bad',  'sagittal'),
        (os.path.join(BASE_DIR, 'JC-axiales', 't2-t21.00.nii'),                      'JC',            'axial'),
        (os.path.join(BASE_DIR, 'JC-axiales', 't2-t22.00.nii'),                      'JC',            'axial'),
        (os.path.join(BASE_DIR, 'JC-axiales', 't2-t23.00.nii'),                      'JC',            'axial'),
        (os.path.join(BASE_DIR, 'JC-axiales', 't2-t24.00.nii'),                      'JC',            'axial'),
        (os.path.join(BASE_DIR, 'JC-axiales', 't2-t25.00.nii'),                      'JC',            'axial'),
    ],

    # ---- ORIENTACIÓN POR ARCHIVO (clave = subcadena del id) ----
    # fix = (rot_k, flip_h, flip_v).  rot_k = nº de giros de 90° (0,1,2,3).
    # Objetivo: TODOS los cortes con los VENTRÍCULOS HACIA ABAJO.
    # Si algún volumen sale girado al revés, cambia rot_k (p.ej. 1 -> 3) o pon flip_v=True.
    'orient_fix': {
        '56-good':     (2, False, False),   # girar (user requested 90 deg more, so 180 total)
        '49-badrecon': (0, False, False),   # girar (user requested 90 deg more, so 180 total)
        '66-badrecon': (1, False, False),   # girar
        '05-badrecon': (1, False, False),   # girar
        '96-badrecon': (2, False, False),   # User requested 180 degree rotation
        't2-t21':      (0, False, False),
        't2-t22':      (0, False, False),
        't2-t23':      (0, False, False),
        't2-t24':      (0, False, False),
        't2-t25':      (0, False, False),
    },
    # Reconstrucciones de baja calidad: aplicar Non-Local Means antes de segmentar
    'nlm_files': ['49-badrecon', '66-badrecon', '05-badrecon', '96-badrecon'],
    'auto_orient': True,   # respaldo automatico si un archivo no tiene fix manual

    'pct_low': 0.5,   'pct_high': 99.5,
    'brain_thresh': 0.05,  'roi_margin': 0.05,
    'z_range_pct': (0.42, 0.62),  'score_sigma': 2,
    'csf_pct_low': 75,  'csf_pct_high': 95,
    'clahe_clip': 0.005,  'clahe_nbins': 256,
    'sigma_normal': 1.0,  'sigma_lowsnr': 2.0,  'snr_thresh': 6.0,
    'median_ksize': 3,  'bilateral_d': 9,  'bilateral_sc': 50,  'bilateral_ss': 50,
    'close_r': 3,  'open_r': 2,  'min_size': 50,
    'ventricle_pct': 82,  'max_ventricle_pct': 0.14,
    'dsp_kernel': 2,  'dsp_median_tam': 3,
    'aniso_niter': 8,  'aniso_kappa': 30,  'aniso_gamma': 0.1,
    'min_area_px': 30,  'max_mask_frac': 0.12,
    'cross_slices': 3,  'cv_thresh': 0.25,
    # ---- CONTROL ANTI-FUGA de la segmentacion (mantiene mascaras compactas) ----
    # Topes calibrados para que el VENTRICULO COMPLETO (asta frontal + cuerpo +
    # atrio) se forme como banda curva (estilo figura de referencia t2-t24),
    # sin fugarse a medio hemisferio. Si algo se fuga: baja max_grow_frac/max_seed_pct.
    # Si queda muy chico: sube grow_floor_pct (menos restrictivo = mas crecimiento).
    # --- Parametros de segmentacion ventricular (CALIBRADOS con datos reales JC) ---
    'seed_pct': 91,             # percentil de la semilla DENTRO de la region interna
    'inner_erode': 0.18,        # erosion del cerebro para excluir LCR subaracnoideo
    'atrium_post_frac': 0.62,   # prioriza el atrio (mitad posterior-inferior)
    'max_seed_pct': 0.07,       # area maxima de la semilla (frac. del cerebro)
    'max_grow_frac': 0.05,      # tope de crecimiento del ventriculo (acotado)
    'grow_floor_pct': 85,       # percentil minimo de crecimiento
    'leak_factor': 2.3,         # tope de salto de area por paso (anti-fuga)
    'midline_excl_frac': 0.045, # banda central excluida (cavum / 3er ventriculo)
    'target_size': (256, 256),  'batch_size': 2,
    'cnn_epochs': 10,  'cnn_lr': 1e-4,  'encoder_name': 'efficientnet-b0',
    'mild_mm': 10.0,  'moderate_mm': 12.5,  'severe_mm': 15.0,
}

RESULTADOS_DIR = os.path.join(BASE_DIR, 'resultados')
os.makedirs(RESULTADOS_DIR, exist_ok=True)

ANISO_NITER = CONFIG['aniso_niter']
ANISO_KAPPA = CONFIG['aniso_kappa']
ANISO_GAMMA = CONFIG['aniso_gamma']

print('OK  CONFIG cargado.')
print(f'    Archivos configurados: {len(CONFIG["archivos"])}')
print(f'    Orientacion por archivo: {len(CONFIG["orient_fix"])} entradas')
print(f'    96-badrecon orient_fix: {CONFIG["orient_fix"]["96-badrecon"]}')
print(f'    49-badrecon orient_fix: {CONFIG["orient_fix"]["49-badrecon"]}')
print(f'    Resultados en: {RESULTADOS_DIR}')

# ---- VERIFICACIÓN DE RUTAS EN DRIVE ----
print('\nVerificando archivos en Google Drive...')
if not os.path.isdir(BASE_DIR):
    print(f'  ERROR: no se encontró la carpeta base:\n    {BASE_DIR}')
    print('  Revisa que "Ejemplos mri" esté en la raíz de MiUnidad,')
    print('  o ajusta BASE_DIR arriba con la ruta correcta.')
else:
    n_ok = 0
    for ruta, tipo, plano in CONFIG['archivos']:
        if os.path.isfile(ruta):
            print(f'  OK        {os.path.relpath(ruta, BASE_DIR)}'); n_ok += 1
        else:
            print(f'  FALTANTE  {os.path.relpath(ruta, BASE_DIR)}')
    print(f'\n  {n_ok}/{len(CONFIG["archivos"])} archivos encontrados en Drive.')


In [ ]:
# SECCIÓN A — Carga de volúmenes, tabla inventario y mosaico (orientado)
# (orientacion inline: esta celda corre ANTES de definir las funciones v3)
volumenes = []
filas_tabla = []

def _orient_local(sl, fix):
    rot_k, fh, fv = fix
    o = np.rot90(sl, k=int(rot_k) % 4)
    if fh: o = np.flip(o, axis=0)
    if fv: o = np.flip(o, axis=1)
    return np.ascontiguousarray(o)

def _auto_local(sl):
    try:
        th = threshold_otsu(sl)
    except Exception:
        return (0, False, False)
    m = remove_small_objects(binary_fill_holes(sl > th), min_size=100)
    if m.sum() == 0: return (0, False, False)
    rr, cc = np.where(m); w = rr.max()-rr.min(); h = cc.max()-cc.min()
    return (1, False, False) if w > h*1.15 else (0, False, False)

def _match_fix(nombre, cfg):
    for k, v in cfg['orient_fix'].items():
        if k in nombre: return v
    return None

for ruta, tipo, plano in CONFIG['archivos']:
    nombre = Path(ruta).stem.replace('.00', '')
    if not os.path.exists(ruta):
        print(f'AVISO: No encontrado — {ruta}')
        continue
    try:
        img   = nib.load(ruta)
        img   = nib.as_closest_canonical(img)
        data  = img.get_fdata().squeeze().astype(np.float32)
        zooms = img.header.get_zooms()
        sx, sy, sz = float(zooms[0]), float(zooms[1]), float(zooms[2])
        ori   = nib.aff2axcodes(img.affine)
        n_slices_display = data.shape[0] if plano=='sagittal' else data.shape[2]

        fix = _match_fix(nombre, CONFIG)
        use_nlm = any(k in nombre for k in CONFIG['nlm_files'])

        print(f'\nDATOS DEL VOLUMEN: {nombre}')
        print(f'├─ Tipo : {tipo}   Plano : {plano}   OrientFix : {fix}   NLM : {use_nlm}')
        print(f'├─ Forma: {data.shape}   Espaciado(mm): {sx:.2f}x{sy:.2f}x{sz:.2f}')
        print(f'├─ Rango: [{data.min():.1f}, {data.max():.1f}]   Cortes: 0-{n_slices_display-1}')
        print(f'└─ Orientacion NIfTI: {ori}')

        volumenes.append({'id': nombre, 'tipo': tipo, 'plano': plano, 'ruta': ruta,
                          'data_raw': data, 'spacing': (sx, sy, sz),
                          'n_slices': n_slices_display, 'orientacion': ori,
                          'orient_fix': fix, 'nlm': use_nlm})
        filas_tabla.append({'ID': nombre, 'Tipo': tipo, 'Plano': plano,
                            'Forma': str(data.shape),
                            'Espaciado': f'{sx:.2f}x{sy:.2f}x{sz:.2f}',
                            'OrientFix': str(fix), 'NLM': use_nlm,
                            'N_cortes': n_slices_display})
    except Exception as e:
        print(f'ERROR al cargar {ruta}: {e}')

print(f'\n{"="*55}\n  Total de volúmenes cargados: {len(volumenes)}\n{"="*55}')
if len(volumenes) == 0:
    print('\n' + '!'*64)
    print('  ERROR CRITICO: 0 volúmenes cargados.')
    print('  Las rutas de CONFIG["archivos"] no existen en este entorno.')
    print('  Soluciones:')
    print('   1) Monta Google Drive (celda de drive.mount) y verifica que la')
    print('      carpeta BASE_DIR sea correcta:')
    print('        ', BASE_DIR)
    print('   2) Si tus .nii están en otra ruta, edita BASE_DIR o CONFIG["archivos"].')
    print('   3) Revisa arriba las líneas "FALTANTE" para ver qué archivo no se halló.')
    print('  Las celdas siguientes NO funcionarán hasta cargar al menos 1 volumen.')
    print('!'*64 + '\n')

df_inv = pd.DataFrame(filas_tabla)
display(df_inv)
df_inv.to_csv(os.path.join(RESULTADOS_DIR, 'A_inventario.csv'), index=False)

# Mosaico del corte central YA ORIENTADO (ventriculos abajo)
n = len(volumenes)
if n > 0:
    cols = min(5, n); rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
    fig.patch.set_facecolor('#0d1b2a'); axes = np.array(axes).flatten()
    tipo_color = {'hospital_good':'limegreen', 'hospital_bad':'tomato', 'JC':'deepskyblue'}
    for i, vol in enumerate(volumenes):
        mid_idx = (vol['data_raw'].shape[0] if vol['plano']=='sagittal'
                   else vol['data_raw'].shape[2]) // 2
        dn = (np.clip(vol['data_raw'], np.percentile(vol['data_raw'],0.5), np.percentile(vol['data_raw'],99.5))
              - np.percentile(vol['data_raw'],0.5)) / (np.percentile(vol['data_raw'],99.5)-np.percentile(vol['data_raw'],0.5)+1e-8)
        sl0 = dn[mid_idx,:,:] if vol['plano']=='sagittal' else dn[:,:,mid_idx]
        fixv = vol['orient_fix'] if vol['orient_fix'] is not None else (_auto_local(sl0) if CONFIG.get('auto_orient',True) else (0,False,False))
        vol['orient_fix'] = fixv
        corte = _orient_local(sl0, fixv)
        axes[i].imshow(corte.T, cmap='gray', origin='lower',
                       vmin=np.percentile(corte,0.5), vmax=np.percentile(corte,99.5))
        axes[i].set_title(f'{vol["id"]}\n({vol["tipo"]})', fontsize=8,
                          color=tipo_color.get(vol['tipo'], 'white'))
        axes[i].set_facecolor('#0d1b2a'); axes[i].axis('off')
    for j in range(n, len(axes)): axes[j].axis('off')
    plt.suptitle('Inventario — Corte central ORIENTADO (ventriculos abajo)',
                 fontsize=13, color='white')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTADOS_DIR, 'A_inventario_mosaico.png'), dpi=130, facecolor='#0d1b2a')
    plt.show()

---
## Sección B — Explorador Interactivo de Volúmenes
Widget con Dropdown + IntSlider. El slider actualiza su rango al cambiar de volumen.

In [ ]:
# SECCIÓN B — Explorador interactivo de volúmenes con ipywidgets
if len(volumenes) == 0:
    print('AVISO: No hay volúmenes cargados.')
else:
    opciones  = [f'{v["id"]} ({v["tipo"]})' for v in volumenes]
    _vol_map  = {f'{v["id"]} ({v["tipo"]})': v for v in volumenes}

    dropdown  = widgets.Dropdown(options=opciones, value=opciones[0],
                                  description='Volumen:', style={'description_width':'80px'},
                                  layout=widgets.Layout(width='420px'))
    slider    = widgets.IntSlider(min=0, max=volumenes[0]['n_slices']-1,
                                   value=volumenes[0]['n_slices']//2,
                                   description='Corte Z:', style={'description_width':'70px'},
                                   layout=widgets.Layout(width='420px'))
    output    = widgets.Output()

    def _mostrar_corte(nombre_vol, z_idx):
        vol   = _vol_map[nombre_vol]
        corte = vol['data_raw'][:, :, z_idx].T.astype(float)
        vmin  = np.percentile(corte, 0.5); vmax = np.percentile(corte, 99.5)
        cn    = np.clip((corte - vmin)/(vmax - vmin + 1e-8), 0, 1)
        fg    = cn[cn > 0.05]
        snr   = float(fg.mean()/(fg.std()+1e-8)) if len(fg)>10 else 0.0
        pct95 = float(np.percentile(cn, 95))

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
        ax1.imshow(cn, cmap='gray', origin='lower')
        ax1.set_title(f'Archivo: {vol["id"]} | Corte Z={z_idx} | SNR={snr:.1f}', fontsize=9)
        ax1.axis('off')
        ax2.hist(cn.flatten(), bins=80, color='steelblue', alpha=0.7)
        ax2.axvline(pct95, color='red', lw=1.5, ls='--', label=f'Pct95={pct95:.2f}')
        ax2.set_xlabel('Intensidad normalizada'); ax2.set_ylabel('Frecuencia')
        ax2.set_title(f'Histograma | Rango [{corte.min():.0f}, {corte.max():.0f}]')
        ax2.legend(fontsize=8)
        plt.suptitle(f'{vol["id"]} ({vol["tipo"]}) — Forma:{vol["data_raw"].shape} '
                     f'Espaciado:{vol["spacing"][0]:.2f}mm/px', fontsize=10)
        plt.tight_layout(); plt.show()

    def _on_vol_change(change):
        vol = _vol_map[change['new']]
        slider.max = vol['n_slices'] - 1; slider.value = vol['n_slices'] // 2

    def _update(change):
        with output:
            clear_output(wait=True); _mostrar_corte(dropdown.value, slider.value)

    dropdown.observe(_on_vol_change, names='value')
    dropdown.observe(_update, names='value')
    slider.observe(_update, names='value')
    display(widgets.VBox([dropdown, slider, output]))
    _update(None)

---
## Sección C — Etapa I: Análisis Digital
Análisis cuantitativo: dimensiones, resolución, FOV, Nyquist y estadísticas.
Visualización con 6 transformaciones (ecualización, gamma, log).

> Ejecutar toda la **Sección D** antes de correr la celda de ejecución de esta sección.

In [ ]:
def etapa_1_analisis_digital(vol, z_idx):
    # Análisis digital cuantitativo y visual de un corte MRI (orientado, plano-aware)
    plano = vol.get('plano', 'axial')
    corte_raw = (vol['data_raw'][z_idx, :, :] if plano == 'sagittal'
                 else vol['data_raw'][:, :, z_idx])
    fix = vol.get('orient_fix') or (0, False, False)
    corte_raw = apply_orientation(corte_raw, fix)
    sx, sy, sz = vol['spacing']
    if int(fix[0]) % 2 == 1: sx, sy = sy, sx
    rows, cols = corte_raw.shape

    res_bits  = math.ceil(math.log2(max(corte_raw.max() + 1, 2)))
    fov_x = rows * sx; fov_y = cols * sy
    nyquist_x = 1.0/(2.0*sx); nyquist_y = 1.0/(2.0*sy)
    metricas = {
        'Tamanio de matriz': f'{rows} x {cols} px',
        'Resolucion espacial': f'{sx:.3f} x {sy:.3f} mm/px',
        'Resolucion de intensidad': f'{res_bits} bits',
        'FOV x': f'{fov_x:.1f} mm', 'FOV y': f'{fov_y:.1f} mm',
        'Frecuencia Nyquist x': f'{nyquist_x:.3f} c/mm',
        'Frecuencia Nyquist y': f'{nyquist_y:.3f} c/mm',
        'Media de intensidad': f'{corte_raw.mean():.2f}',
        'Desviacion estandar': f'{corte_raw.std():.2f}',
        'Minimo': f'{corte_raw.min():.2f}', 'Maximo': f'{corte_raw.max():.2f}'}
    df_m = pd.DataFrame(list(metricas.items()), columns=['Parametro', 'Valor'])
    print(f'\n{"="*50}\n  ANALISIS DIGITAL — {vol["id"]}  (corte idx={z_idx})\n{"="*50}')
    display(df_m)

    vmin = np.percentile(corte_raw, 0.5); vmax = np.percentile(corte_raw, 99.5)
    c_norm = np.clip((corte_raw - vmin)/(vmax - vmin + 1e-8), 0, 1)
    c_u8 = (c_norm * 255).astype(int)
    c_eq    = ecualizar_histograma(c_u8)
    c_gamma = transformacion_gamma(c_u8, gamma=0.5, c=1.0)
    c_log   = transformacion_log(c_u8, c=1.0)

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes[0,0].imshow(c_norm.T, cmap='gray', origin='lower'); axes[0,0].set_title(f'Original orientado (idx={z_idx})')
    axes[0,1].hist(c_norm.flatten(), bins=80, color='steelblue', alpha=0.7)
    axes[0,1].set_title('Histograma de intensidades'); axes[0,1].set_xlabel('Intensidad'); axes[0,1].set_ylabel('Frecuencia')
    axes[0,2].imshow(c_eq.T, cmap='gray', origin='lower'); axes[0,2].set_title('Ecualizacion histograma (prof.)')
    axes[1,0].hist(c_eq.flatten(), bins=80, color='darkorange', alpha=0.7)
    axes[1,0].set_title('Histograma ecualizado'); axes[1,0].set_xlabel('Intensidad'); axes[1,0].set_ylabel('Frecuencia')
    axes[1,1].imshow(c_gamma.T, cmap='gray', origin='lower'); axes[1,1].set_title('Gamma (0.5) — realce oscuros')
    axes[1,2].imshow(c_log.T, cmap='gray', origin='lower'); axes[1,2].set_title('Logaritmica — expansion bajos')
    for ax in [axes[0,0], axes[0,2], axes[1,1], axes[1,2]]: ax.axis('off')
    plt.suptitle(f'Etapa I — Analisis Digital · {vol["id"]} ({vol["tipo"]})', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTADOS_DIR, f'C_etapa1_{vol["id"]}.png'), dpi=100)
    plt.show()
    return df_m


---
## Sección D — Funciones DSP del Profesor (Implementación Manual)

Implementaciones exactas de los algoritmos del profesor. Los bucles explícitos
de `filtro_mediana`, `erosion`, `dilatacion`, `eliminar_objetos_pequenos` y
`etiquetar_componentes` son **lentos en imágenes grandes** — se aplican sobre
recortes de 128×128 px para las comparaciones en Sección E.

In [ ]:
# FUNCIONES DSP DEL PROFESOR — IMPLEMENTACIÓN MANUAL (conservar exactas)
# Ref: Pipeline_Completo_MRI.ipynb — Tecnológico de Monterrey

# Convolución 2D base con bucles explícitos
def filtro_2d(A, w):
    A=np.array(A,dtype=float); w=np.array(w,dtype=float)
    N,M=A.shape; nw=w.shape[0]
    Ap=np.zeros((N+nw-1,M+nw-1)); Ap[1:N+1,1:M+1]=A
    B=np.zeros((N,M))
    for n in range(N):
        for m in range(M):
            for i in range(-1,2):
                for j in range(-1,2):
                    B[n,m]+=w[i+1,j+1]*Ap[n-i+1,m-j+1]
    return B

# ------------------------------------------------------------------
# BANCO DE KERNELS (mascaras de convolucion 3x3).
# JUSTIFICACION DE LA ELECCION DE KERNEL (pregunta tipica del jurado):
#   - n=1 Promedio (caja 1/9): suaviza fuerte pero DESTRUYE bordes -> NO para LCR.
#   - n=2 Gaussiano ponderado (centro pesa mas): suavizado lineal que respeta
#         gradientes -> ELEGIDO para ruido Riciano (no impulsivo) y para dejar
#         transiciones LCR/tejido limpias antes de umbralizar.
#   - n=3,4 Sobel direccional (Gx, Gy "simples").
#   - n=5,6 Sobel ponderado (centro x2) -> mejor relacion señal/ruido en el borde,
#         por eso bordes_sobel() usa 5 y 6.
#   - n=7,8 Laplaciano (realce de 2da derivada) -> NO se usa en el pipeline final
#         porque amplifica ruido (ver seccion "Por que NO se usan" de la prese).
# El tamaño 3x3 se elige porque el ventriculo fetal mide pocos px: kernels mas
# grandes (5x5, 7x7) difuminarian el atrio y subestimarian el diametro atrial.
# ------------------------------------------------------------------
def filtro_pro(A, n):
    kernels={
        1:np.ones((3,3))/9,
        2:np.array([[0.36,0.6,0.36],[0.6,1.0,0.6],[0.36,0.6,0.36]])/4.8976,
        3:np.array([[1,0,-1],[1,0,-1],[1,0,-1]],dtype=float),
        4:np.array([[-1,-1,-1],[0,0,0],[1,1,1]],dtype=float),
        5:np.array([[1,0,-1],[2,0,-2],[1,0,-1]],dtype=float),
        6:np.array([[1,2,1],[0,0,0],[-1,-2,-1]],dtype=float),
        7:np.array([[0,1,0],[1,-4,1],[0,1,0]],dtype=float),
        8:np.array([[1,1,1],[1,-8,1],[1,1,1]],dtype=float),
    }
    return filtro_2d(A,kernels[n])

# Filtro de mediana con bucles explícitos
def filtro_mediana(imagen, tam=3):
    imagen=np.array(imagen,dtype=float); N,M=imagen.shape; k=tam//2
    resultado=np.zeros_like(imagen)
    for i in range(N):
        for j in range(M):
            v=imagen[max(0,i-k):min(N,i+k+1),max(0,j-k):min(M,j+k+1)]
            resultado[i,j]=np.median(v)
    return resultado

# Gradiente Sobel: magnitud y componentes
def bordes_sobel(imagen):
    gx=filtro_pro(imagen,5); gy=filtro_pro(imagen,6)
    return np.sqrt(gx**2+gy**2),gx,gy

# Error cuadrático medio
def mse(original,filtrada):
    return np.mean((np.array(original,dtype=float)-np.array(filtrada,dtype=float))**2)

# Ecualización de histograma con bucles
def ecualizar_histograma(imagen):
    imagen=np.array(imagen,dtype=int); N,M=imagen.shape
    h=np.zeros(256)
    for v in imagen.flatten():
        if 0<=v<=255: h[v]+=1
    cdf=np.cumsum(h); cdf_n=(cdf-cdf.min())/(N*M-cdf.min()+1e-8)*255
    eq=np.zeros_like(imagen)
    for i in range(N):
        for j in range(M): eq[i,j]=int(cdf_n[imagen[i,j]])
    return eq.astype(float)

# Transformaciones tonales
def transformacion_gamma(imagen,gamma=1.0,c=1.0):
    return np.clip(c*(np.array(imagen,dtype=float)/255.0)**gamma*255,0,255)

def transformacion_log(imagen,c=1.0):
    r=c*np.log1p(np.array(imagen,dtype=float)/255.0)
    return np.clip(r/r.max()*255,0,255) if r.max()>0 else np.zeros_like(imagen,dtype=float)

# Elementos estructurantes
def crear_se(tipo='cruz',tam=3):
    r=tam//2; se=np.zeros((tam,tam),dtype=int)
    if tipo=='cruz': se[r,:]=1; se[:,r]=1
    elif tipo=='cuadrado': se[:,:]=1
    elif tipo=='disco':
        for i in range(tam):
            for j in range(tam):
                if (i-r)**2+(j-r)**2<=r**2: se[i,j]=1
    return se

# Erosión morfológica con bucles
def erosion(A,se=None):
    if se is None: se=crear_se('cruz',3)
    A=np.array(A,dtype=float); N,M=A.shape; k=se.shape[0]//2; B=np.zeros_like(A)
    for i in range(k,N-k):
        for j in range(k,M-k):
            if np.all(A[i-k:i+k+1,j-k:j+k+1][se==1]==1): B[i,j]=1
    return B

# Dilatación morfológica con bucles
def dilatacion(A,se=None):
    if se is None: se=crear_se('cruz',3)
    A=np.array(A,dtype=float); N,M=A.shape; k=se.shape[0]//2; B=np.zeros_like(A)
    for i in range(k,N-k):
        for j in range(k,M-k):
            if np.any(A[i-k:i+k+1,j-k:j+k+1][se==1]==1): B[i,j]=1
    return B

def opening(A,se=None): return dilatacion(erosion(A,se),se)
def closing(A,se=None): return erosion(dilatacion(A,se),se)
def gradiente_morfologico(A,se=None): return dilatacion(A,se)-erosion(A,se)

# Eliminar objetos pequeños con BFS
def eliminar_objetos_pequenos(binaria,min_area=50):
    binaria=np.array(binaria,dtype=int).copy(); N,M=binaria.shape
    etiq=np.zeros_like(binaria); label=0
    for i in range(N):
        for j in range(M):
            if binaria[i,j]==1 and etiq[i,j]==0:
                label+=1; cola=[(i,j)]; pix=[]
                while cola:
                    ci,cj=cola.pop(0)
                    if 0<=ci<N and 0<=cj<M and binaria[ci,cj]==1 and etiq[ci,cj]==0:
                        etiq[ci,cj]=label; pix.append((ci,cj))
                        for di,dj in [(-1,0),(1,0),(0,-1),(0,1)]: cola.append((ci+di,cj+dj))
                if len(pix)<min_area:
                    for pi,pj in pix: binaria[pi,pj]=0
    return binaria

def binarizar(imagen,threshold): return (imagen>threshold).astype(float)

# Umbral de Otsu con cálculo explícito
def umbral_otsu(imagen):
    imagen=np.array(imagen,dtype=int); hist=np.zeros(256)
    for v in imagen.flatten():
        if 0<=v<=255: hist[v]+=1
    total=imagen.size; prob=hist/total
    mejor_var,T_opt=0,0
    for T in range(256):
        w0=np.sum(prob[:T+1]); w1=np.sum(prob[T+1:])
        if w0==0 or w1==0: continue
        mu0=np.sum(np.arange(T+1)*prob[:T+1])/w0
        mu1=np.sum(np.arange(T+1,256)*prob[T+1:])/w1
        var=w0*w1*(mu0-mu1)**2
        if var>mejor_var: mejor_var,T_opt=var,T
    return T_opt

# K-means simple: devuelve cluster de mayor intensidad
def kmeans_simple(imagen,k=2,max_iter=20):
    datos=imagen.flatten()
    centroides=np.linspace(datos.min(),datos.max(),k)
    for _ in range(max_iter):
        dist=np.abs(datos[:,None]-centroides[None,:]); etiq=np.argmin(dist,axis=1)
        nuevos=np.array([np.mean(datos[etiq==c]) if np.any(etiq==c) else centroides[c] for c in range(k)])
        if np.allclose(centroides,nuevos,atol=0.1): break
        centroides=nuevos
    return (etiq==np.argmax(centroides)).reshape(imagen.shape).astype(float)

# Etiquetado de componentes con BFS
def etiquetar_componentes(binaria):
    binaria=np.array(binaria,dtype=int); N,M=binaria.shape; etiq=np.zeros_like(binaria); n=0
    for i in range(N):
        for j in range(M):
            if binaria[i,j]==1 and etiq[i,j]==0:
                n+=1; cola=[(i,j)]
                while cola:
                    ci,cj=cola.pop(0)
                    if 0<=ci<N and 0<=cj<M and binaria[ci,cj]==1 and etiq[ci,cj]==0:
                        etiq[ci,cj]=n
                        for di,dj in [(-1,0),(1,0),(0,-1),(0,1)]: cola.append((ci+di,cj+dj))
    return etiq,n

# FFT 1D usando Cooley-Tukey dividido en pares e impares
def fft_1d(f):
    f=np.array(f,dtype=complex).flatten(); N=len(f); K=N//2; x=np.arange(K)
    Fp=np.zeros(K,dtype=complex); Fn=np.zeros(K,dtype=complex); F1=np.zeros(N,dtype=complex)
    for u in range(K):
        Wk=np.exp(-1j*2*np.pi*u*x/K); Wk2=np.exp(-1j*np.pi*u/K)
        Fp[u]=np.dot(Wk,f[2*x]); Fn[u]=np.dot(Wk,f[2*x+1])
        F1[u]=Fp[u]+Fn[u]*Wk2; F1[u+K]=Fp[u]-Fn[u]*Wk2
    return F1

# FFT 2D: aplica fft_1d por filas y columnas
def fft_2d(A):
    A=np.array(A,dtype=complex); N,M=A.shape
    F1=np.zeros((N,M),dtype=complex); F2=np.zeros((N,M),dtype=complex)
    for y in range(M): F1[:,y]=fft_1d(A[:,y])
    for x in range(N): F2[x,:]=fft_1d(F1[x,:])
    return F2

def ifft_1d(f):
    f=np.array(f,dtype=complex).flatten(); N=len(f); K=N//2; x=np.arange(K)
    Fp=np.zeros(K,dtype=complex); Fn=np.zeros(K,dtype=complex); F1=np.zeros(N,dtype=complex)
    for u in range(K):
        Wk=np.exp(1j*2*np.pi*u*x/K); Wk2=np.exp(1j*np.pi*u/K)
        Fp[u]=np.dot(Wk,f[2*x]); Fn[u]=np.dot(Wk,f[2*x+1])
        F1[u]=Fp[u]+Fn[u]*Wk2; F1[u+K]=Fp[u]-Fn[u]*Wk2
    return F1

def ifft_2d(A):
    A=np.array(A,dtype=complex); N,M=A.shape
    F1=np.zeros((N,M),dtype=complex); f=np.zeros((N,M),dtype=complex)
    for y in range(M): F1[:,y]=ifft_1d(A[:,y])
    for x in range(N): f[x,:]=ifft_1d(F1[x,:])
    return f/(N*M)

# Filtro pasa-bajas en el dominio de la frecuencia
def pasa_bajas(P,Q,D0):
    v,u=np.meshgrid(np.arange(1,Q+1),np.arange(1,P+1))
    return (np.sqrt((u-P/2)**2+(v-Q/2)**2)<=D0).astype(float)

# Difusión anisótropa Perona-Malik (1990) — preserva bordes ventriculares
def anisotropic_diffusion(img, niter=ANISO_NITER, kappa=ANISO_KAPPA, gamma=ANISO_GAMMA):
    # niter: iteraciones | kappa: conductancia (menor=mas bordes) | gamma: paso (<=0.25)
    img = img.astype(np.float64)
    for _ in range(niter):
        dN = np.roll(img, -1, axis=0) - img
        dS = np.roll(img,  1, axis=0) - img
        dE = np.roll(img, -1, axis=1) - img
        dW = np.roll(img,  1, axis=1) - img
        cN = np.exp(-(dN/kappa)**2); cS = np.exp(-(dS/kappa)**2)
        cE = np.exp(-(dE/kappa)**2); cW = np.exp(-(dW/kappa)**2)
        img += gamma*(cN*dN+cS*dS+cE*dE+cW*dW)
    return img.astype(np.float32)

# Overlay: superposición de máscara en rojo sobre imagen gris
def construir_overlay(corte,mask):
    ov=np.stack([corte/max(corte.max(),1)]*3,axis=-1)
    ov[:,:,0]=np.where(mask==1,1.0,ov[:,:,0])
    ov[:,:,1]=np.where(mask==1,0.2,ov[:,:,1])
    ov[:,:,2]=np.where(mask==1,0.2,ov[:,:,2])
    return ov

print('OK  Funciones DSP del profesor cargadas (26 funciones).')

In [ ]:
# FUNCIONES MODERNAS V3 — Capa adicional sobre las del profesor
# Ref: Analyzer1_v3_clean.ipynb — Tecnológico de Monterrey

def normalize_percentile(vol, plow=0.5, phigh=99.5):
    # Normalización percentil: recorta extremos y escala a [0,1]
    vmin = np.percentile(vol, plow); vmax = np.percentile(vol, phigh)
    return (np.clip(vol, vmin, vmax) - vmin) / (vmax - vmin + 1e-8)

def get_brain_roi(vol_norm, threshold=0.05, margin=0.05):
    # ROI automático: proyección máxima con margen
    proj = vol_norm.max(axis=0); brain = proj > threshold
    coords = np.where(brain)
    if len(coords[0]) == 0: return 0, vol_norm.shape[1], 0, vol_norm.shape[2]
    y0, y1 = coords[0].min(), coords[0].max(); z0, z1 = coords[1].min(), coords[1].max()
    dy = int((y1-y0)*margin); dz = int((z1-z0)*margin)
    return max(0,y0-dy), min(vol_norm.shape[1],y1+dy), max(0,z0-dz), min(vol_norm.shape[2],z1+dz)

def compute_snr(slice_2d):
    # SNR = media/std de píxeles de primer plano
    fg = slice_2d[slice_2d > 0.05]
    return float(fg.mean()/(fg.std()+1e-8)) if len(fg) > 10 else 0.0

def preprocess_slice_v3(slice_2d, cfg):
    # Pipeline v3: MedianBlur -> CLAHE -> Gaussiano (sigma adaptativo) -> Bilateral
    snr    = compute_snr(slice_2d)
    sigma  = cfg['sigma_normal'] if snr >= cfg['snr_thresh'] else cfg['sigma_lowsnr']
    img_u8 = (slice_2d * 255).astype(np.uint8)
    den    = cv2.medianBlur(img_u8, cfg['median_ksize']).astype(np.float64) / 255.0
    clahe  = exposure.equalize_adapthist(den, clip_limit=cfg['clahe_clip'], nbins=cfg['clahe_nbins'])
    gauss  = gaussian_filter(clahe, sigma=sigma)
    g_u8   = (gauss * 255).astype(np.uint8)
    bil    = cv2.bilateralFilter(g_u8, d=cfg['bilateral_d'],
                                 sigmaColor=cfg['bilateral_sc'], sigmaSpace=cfg['bilateral_ss'])
    return bil.astype(np.float64) / 255.0, snr, sigma

def auto_orient_slice(slice_2d):
    # Corrige orientación: frontal al norte (arriba)
    thresh = threshold_otsu(slice_2d)
    mask   = remove_small_objects(binary_fill_holes(slice_2d > thresh), min_size=200)
    if mask.sum() == 0: return slice_2d, 0.0
    rows, cols = np.where(mask)
    brain_w = rows.max()-rows.min(); brain_h = cols.max()-cols.min()
    correction = 0.0; rotated = slice_2d.copy()
    if brain_w > brain_h * 1.2:
        correction = 90.0
        rotated = rotate(slice_2d, 90, resize=False, preserve_range=True, mode='constant', cval=0)
    thresh2 = threshold_otsu(rotated)
    mask2   = binary_fill_holes(remove_small_objects(rotated > thresh2, min_size=100))
    if mask2.sum() == 0: return rotated, correction
    rows2, cols2 = np.where(mask2)
    col_min2, col_max2 = cols2.min(), cols2.max(); brain_h2 = col_max2-col_min2
    if brain_h2 < 10: return rotated, correction
    band  = max(2, brain_h2//4)
    top_w = int(mask2[:, col_max2-band:col_max2].sum())
    bot_w = int(mask2[:, col_min2:col_min2+band].sum())
    if top_w > bot_w * 1.05:
        rotated = np.flip(rotated, axis=1).copy(); correction = (correction+180)%360
    return rotated, correction

def detect_midline(slice_proc, brain_mask_2d):
    # Detecta cisura interhemisférica con 3 anclas: centroide + muescas frontal/trasera
    brain_coords = np.where(brain_mask_2d)
    if len(brain_coords[0]) == 0:
        mid = slice_proc.shape[0]//2; return mid, None, None, mid-5, mid+5, mid, mid
    brain_cols = brain_coords[0]; brain_rows = brain_coords[1]
    centroid_x = int(brain_cols.mean())
    col_min, col_max = brain_cols.min(), brain_cols.max()
    row_min, row_max = brain_rows.min(), brain_rows.max()
    brain_width = col_max-col_min; brain_height = row_max-row_min
    margin  = int(brain_width*0.12)
    s_start = max(0, centroid_x-margin); s_end = min(slice_proc.shape[0], centroid_x+margin)
    n_cols  = slice_proc.shape[0]
    col_profile = np.zeros(n_cols)
    for c in range(n_cols):
        px = slice_proc[c, :][brain_mask_2d[c, :]]
        col_profile[c] = px.mean() if len(px) > 5 else 1.0
    col_smooth  = gaussian_filter(col_profile, sigma=2)
    def notch_in_strip(r0, r1):
        strip = slice_proc[s_start:s_end, r0:r1]
        if strip.size == 0: return centroid_x
        return int(np.argmin(strip.mean(axis=1))) + s_start
    front_h     = int(brain_height*0.15); back_h = int(brain_height*0.15)
    notch_front = notch_in_strip(row_max-front_h, row_max)
    notch_back  = notch_in_strip(row_min, row_min+back_h)
    midline_col = int((centroid_x*1 + notch_front*2 + notch_back*2) / 5)
    return midline_col, col_profile, col_smooth, s_start, s_end, notch_front, notch_back

def fill_ventricle_from_seed(seed_mask, slice_proc, brain_mask, cfg):
    # CRECIMIENTO QUE LLENA EL VENTRICULO COMPLETO (asta frontal + cuerpo + atrio)
    # como banda curva, PERO con deteccion de fuga por "salto de area".
    #
    # IDEA: bajamos el umbral gradualmente y dejamos crecer la region conectada a
    # la semilla. Mientras el crecimiento sea SUAVE (la banda se va llenando) lo
    # aceptamos. Si en un paso el area PEGA UN SALTO brusco (factor de fuga) o
    # supera el tope anatomico, ese paso es una FUGA (se conecto al talamo/corteza)
    # -> lo rechazamos y nos quedamos con el ultimo estado bueno.
    #   - 'max_grow_frac'  : tope duro de tamaño (banda completa, no medio hemisferio)
    #   - 'grow_floor_pct' : percentil minimo (mas bajo => llena mas la banda)
    #   - factor de fuga 2.2: un paso no puede mas que ~duplicar el area previa
    if seed_mask.sum() == 0:
        return seed_mask
    filled   = binary_fill_holes(seed_mask)
    brain_px = slice_proc[brain_mask]
    if len(brain_px) < 50:
        return filled
    brain_area = max(int(brain_mask.sum()), 1)
    max_grow   = brain_area * cfg.get('max_grow_frac', 0.085)
    piso_pct   = cfg.get('grow_floor_pct', 74)
    leak_factor = cfg.get('leak_factor', 3.0)
    prev_sum   = int(filled.sum())
    best       = filled.copy()
    # Empezamos a llenar desde un percentil intermedio (entre la semilla alta y el
    # piso) para que la banda se complete; bajamos de a 2 percentiles.
    pct = cfg.get('ventricle_pct', 82) + 2
    for _ in range(16):                       # hasta 16 pasos finos
        pct -= 2
        if pct < piso_pct:
            break
        t_pct     = np.percentile(brain_px, pct)
        candidate = (slice_proc > t_pct) & brain_mask
        labeled_c, n_lab = scipy_label(candidate)
        new_mask = np.zeros_like(filled, dtype=bool)
        for lbl in range(1, n_lab + 1):
            comp = (labeled_c == lbl)
            if (comp & best).any():
                new_mask |= comp
        new_mask = binary_fill_holes(new_mask)
        grow = int(new_mask.sum())
        if grow == 0:
            continue
        # exceso anatomico absoluto -> esto SI es fuga grave: paramos.
        if grow > max_grow:
            break
        # salto brusco -> probablemente fuga: SALTAMOS este paso (no lo aceptamos)
        # pero seguimos bajando, por si un umbral mas fino llena sin fugarse.
        if grow > prev_sum * leak_factor:
            continue
        best     = new_mask
        prev_sum = grow
    # limpieza morfologica: cierre (une el cuerpo del ventriculo) + apertura ligera
    filled = binary_closing(best, disk(cfg['close_r']))
    filled = binary_opening(filled, disk(max(1, cfg.get('open_r', 2) - 1)))
    filled = remove_small_objects(filled, min_size=cfg['min_size'])
    filled = binary_fill_holes(filled)
    # conservar el componente MAYOR (un solo ventriculo por hemisferio)
    lab2, n2 = scipy_label(filled)
    if n2 > 1:
        sizes = [(lab2 == k).sum() for k in range(1, n2 + 1)]
        filled = (lab2 == int(np.argmax(sizes)) + 1)
    return filled

def get_best_slice(vol, cfg):
    # Returns (best_idx, raw_2d_slice, spacing_2d) for sagittal or axial plane.
    # Uses canonical RAS orientation directly — no auto-rotation needed.
    data_norm = normalize_percentile(vol['data_raw'], cfg['pct_low'], cfg['pct_high'])
    plano = vol.get('plano', 'axial')
    tl = np.percentile(data_norm, cfg['csf_pct_low'])
    th = np.percentile(data_norm, cfg['csf_pct_high'])
    sx, sy, sz = vol['spacing']
    if plano == 'sagittal':
        n = data_norm.shape[0]
        idx_range = list(range(int(n * 0.25), int(n * 0.75)))
        scores = [int(np.sum((data_norm[i,:,:] >= tl) & (data_norm[i,:,:] <= th)))
                  for i in idx_range]
        scores_sm = gaussian_filter(np.array(scores, dtype=float), sigma=cfg['score_sigma'])
        best_i = idx_range[int(np.argmax(scores_sm))]
        sl = data_norm[best_i, :, :]
        sp_2d = (sy, sz)
    else:
        n = data_norm.shape[2]
        # rango de busqueda del plano transventricular (zona media del volumen)
        zr = cfg.get('z_range_pct', (0.35, 0.65))
        z_min = int(n * zr[0]); z_max = int(n * zr[1])
        idx_range = list(range(z_min, z_max))
        # SCORE = par de ventriculos: el mejor corte es aquel donde HAY DOS blobs
        # de LCR interno (uno por hemisferio) y ambos son grandes. Se devuelve el
        # MENOR de los dos mayores -> premia simetria y presencia bilateral.
        # Esto evita cortes altos (solo LCR subaracnoideo) y cortes basales.
        def paired_score(sl_i):
            bmask = sl_i > cfg.get('brain_thresh', 0.05)
            if bmask.sum() < 100: return 0.0
            rr, cc = np.where(bmask)
            er = max(3, int(min(rr.max()-rr.min(), cc.max()-cc.min()) *
                            cfg.get('inner_erode', 0.18)))
            inner = binary_erosion(bmask, disk(er))
            if inner.sum() < 50: return 0.0
            mid = int(cc.mean())
            csf = (sl_i >= tl) & (sl_i <= th) & inner
            csf = remove_small_objects(binary_closing(csf, disk(2)), min_size=40)
            lab, nl = scipy_label(csf)
            if nl < 2: return 0.0
            left, right = [], []
            for k in range(1, nl + 1):
                comp = (lab == k); cx = np.where(comp)[0].mean()
                (left if cx < mid else right).append(int(comp.sum()))
            if not left or not right: return 0.0
            return float(min(max(left), max(right)))
        scores = [paired_score(data_norm[:, :, i]) for i in idx_range]
        scores_sm = gaussian_filter(np.array(scores, dtype=float), sigma=cfg['score_sigma'])
        if scores_sm.max() <= 0:
            # respaldo: si no se hallo par, usar LCR interno total
            best_i = idx_range[len(idx_range) // 2]
        else:
            best_i = idx_range[int(np.argmax(scores_sm))]
        sl = data_norm[:, :, best_i]
        sp_2d = (sx, sy)
    return best_i, sl, sp_2d

def segment_sagittal_ventricle(slice_proc, brain_mask, cfg):
    # Segments lateral ventricle on a SAGITTAL slice.
    # No hemispheric split — finds the single largest bright CSF structure
    # within the inner brain region (excludes peripheral subarachnoid CSF).
    if brain_mask.sum() == 0:
        return np.zeros_like(brain_mask, dtype=bool)
    brain_px = slice_proc[brain_mask]
    if len(brain_px) < 50:
        return np.zeros_like(brain_mask, dtype=bool)
    # Exclude 12% peripheral border from brain bbox (subarachnoid space)
    brain_coords = np.where(brain_mask)
    rmin, rmax = brain_coords[0].min(), brain_coords[0].max()
    cmin, cmax = brain_coords[1].min(), brain_coords[1].max()
    rh = rmax - rmin; cw = cmax - cmin
    inner = np.zeros_like(brain_mask, dtype=bool)
    inner[rmin+int(rh*0.12):rmax-int(rh*0.12),
          cmin+int(cw*0.12):cmax-int(cw*0.12)] = brain_mask[
              rmin+int(rh*0.12):rmax-int(rh*0.12),
              cmin+int(cw*0.12):cmax-int(cw*0.12)]
    # Seed: top ventricle_pct bright pixels within inner brain
    t_seed = np.percentile(brain_px, cfg['ventricle_pct'])
    seed = (slice_proc > t_seed) & inner
    seed = binary_closing(seed, disk(cfg['close_r']))
    seed = remove_small_objects(seed, min_size=cfg['min_size'])
    if seed.sum() == 0:
        return seed
    # Keep only the single largest seed component
    labeled_s, n_lab = scipy_label(seed)
    if n_lab > 1:
        sizes = [(labeled_s == i).sum() for i in range(1, n_lab + 1)]
        seed = (labeled_s == int(np.argmax(sizes)) + 1)
    # Grow to full ventricle
    return fill_ventricle_from_seed(seed, slice_proc, inner, cfg)

def segment_ventricles(slice_proc, brain_mask_2d, midline_col, cfg):
    # ==================================================================
    # SEGMENTACION DE VENTRICULOS LATERALES (validada sobre datos reales JC).
    #
    # CLAVE DEL METODO (por que antes fallaba): en MRI T2 fetal lo MAS brillante
    # NO es el ventriculo, sino el LCR SUBARACNOIDEO periferico (alrededor del
    # cerebro). Si sembramos a percentil alto sobre todo el cerebro, la semilla
    # cae en esa periferia y no en el ventriculo. SOLUCION: erosionar el cerebro
    # para quedarnos con la REGION INTERNA (sin LCR subaracnoideo) y sembrar ahi.
    #
    # PASOS:
    #   1) region interna = erosion del cerebro (quita borde subaracnoideo)
    #   2) por CADA hemisferio (split en la cisura, con banda central excluida):
    #      - semilla = LCR mas brillante (percentil 'seed_pct') dentro de la region
    #      - se prefiere el componente del ATRIO (mitad posterior-inferior)
    #      - se crece de forma acotada (fill_ventricle_from_seed) sin fugarse
    # Devuelve (ventriculo_izquierdo, ventriculo_derecho).
    # ==================================================================
    if brain_mask_2d.sum() == 0:
        return np.zeros_like(brain_mask_2d), np.zeros_like(brain_mask_2d)
    rr, cc = np.where(brain_mask_2d)
    row_min, row_max = rr.min(), rr.max()
    col_min, col_max = cc.min(), cc.max()
    brain_h = row_max - row_min          # eje vertical del display (anterior->posterior)
    brain_w = col_max - col_min          # eje horizontal (izq<->der)
    brain_area = int(brain_mask_2d.sum())

    # (1) REGION INTERNA: erosiona para excluir el LCR subaracnoideo periferico
    er = max(3, int(min(brain_h, brain_w) * cfg.get('inner_erode', 0.18)))
    inner = binary_erosion(brain_mask_2d, disk(er))
    if inner.sum() < 50:
        inner = brain_mask_2d.copy()

    # banda central a excluir (cavum / 3er ventriculo / nucleos mediales brillantes)
    band_excl = max(2, int(brain_w * cfg.get('midline_excl_frac', 0.045)))
    # umbral del atrio: preferimos componentes en la mitad posterior-inferior
    post_limit = row_min + int(brain_h * cfg.get('atrium_post_frac', 0.62))

    def find_one_side(side):
        sm = inner.copy()
        if side == 'L':
            sm[midline_col - band_excl:, :] = False     # conservar lado izquierdo
        else:
            sm[:midline_col + band_excl, :] = False     # conservar lado derecho
        if sm.sum() < 30:
            return np.zeros_like(brain_mask_2d)
        px = slice_proc[sm]
        if len(px) < 30:
            return np.zeros_like(brain_mask_2d)
        thresh = np.percentile(px, cfg.get('seed_pct', 91))
        seed = (slice_proc > thresh) & sm
        seed = binary_closing(seed, disk(cfg['close_r']))
        seed = remove_small_objects(seed, min_size=max(20, cfg['min_size'] // 2))
        labeled, n_lab = scipy_label(seed)
        if n_lab == 0:
            return np.zeros_like(brain_mask_2d)
        # elegir el componente del ATRIO: tamaño ponderado por estar en la zona
        # posterior-inferior (donde se mide el diametro atrial de Cardoza).
        scored = []
        for k in range(1, n_lab + 1):
            comp = (labeled == k)
            cy = np.where(comp)[1].mean()          # posicion antero-posterior
            peso = 1.0 if cy < post_limit else 0.55
            scored.append((comp.sum() * peso, comp))
        scored.sort(key=lambda x: x[0], reverse=True)
        seed1 = scored[0][1]
        # crecimiento acotado dentro de la region interna del hemisferio
        grown = fill_ventricle_from_seed(
            seed1, slice_proc, sm,
            {**cfg, 'ventricle_pct': cfg.get('seed_pct', 91), 'open_r': 1})
        # salvaguarda anti-fuga
        if grown.sum() > brain_area * cfg.get('max_grow_frac', 0.06) * 1.3:
            return binary_fill_holes(seed1)
        return grown

    return find_one_side('L'), find_one_side('R')


def min_bounding_rect_width(binary_mask, spacing_x, spacing_y):
    # Ancho mínimo del rectángulo rotado = diámetro atrial (Cardoza 1988)
    img_u8      = (binary_mask.astype(np.uint8))*255
    contours, _ = cv2.findContours(img_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return 0.0, None
    largest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest) < 10: return 0.0, None
    rect       = cv2.minAreaRect(largest); w_px, h_px = rect[1]
    short_side = min(w_px, h_px)
    return float(short_side * np.mean([spacing_x, spacing_y])), rect

def diagnose(width_mm, cfg):
    # Clasificación de Cardoza 1988 / Pilu 1999
    if   width_mm < cfg['mild_mm']:     return 'Normal',      'limegreen'
    elif width_mm < cfg['moderate_mm']: return 'VM Leve',     'gold'
    elif width_mm < cfg['severe_mm']:   return 'VM Moderada', 'orange'
    else:                               return 'VM Severa',   'red'

def draw_rect_on_ax(ax, rect, color, label):
    # Dibuja el rectángulo mínimo rotado sobre un eje matplotlib
    if rect is None: return
    box  = cv2.boxPoints(rect).astype(int)
    poly = plt.Polygon(box[:, ::-1], fill=False, edgecolor=color, lw=1.5, label=label)
    ax.add_patch(poly)

print('OK  Funciones v3 cargadas (13 funciones: +fill_ventricle, +get_best_slice, +segment_sagittal).')
print('OK  Sección D completa — 39 funciones disponibles.')

# =====================================================================
# CAPA DE ORIENTACIÓN + SEGMENTACIÓN BILATERAL UNIFICADA  (añadido)
# Garantiza: (1) todos los cortes con los ventrículos hacia ABAJO,
#            (2) segmentación L=azul / R=rojo para TODOS los volúmenes,
#            (3) escalera de respaldo para reconstrucciones de baja calidad.
# =====================================================================

def apply_orientation(slice_2d, fix):
    # fix = (rot_k, flip_h, flip_v): rota k*90 deg y voltea segun banderas
    rot_k, flip_h, flip_v = fix
    out = np.rot90(slice_2d, k=int(rot_k) % 4)
    if flip_h: out = np.flip(out, axis=0)
    if flip_v: out = np.flip(out, axis=1)
    return np.ascontiguousarray(out)

def auto_ventricles_down(slice_2d):
    # Heuristica de respaldo: deja el eje largo del cerebro en vertical (display)
    # para que la cisura quede vertical y el split L/R funcione.
    try:
        th = threshold_otsu(slice_2d)
    except Exception:
        return (0, False, False)
    m = remove_small_objects(binary_fill_holes(slice_2d > th), min_size=100)
    if m.sum() == 0: return (0, False, False)
    rr, cc = np.where(m)
    w = rr.max() - rr.min(); h = cc.max() - cc.min()
    return (1, False, False) if w > h * 1.15 else (0, False, False)

def _resolver_fix(vol, sl, cfg):
    fix = vol.get('orient_fix')
    if fix is None:
        fix = auto_ventricles_down(sl) if cfg.get('auto_orient', True) else (0, False, False)
        vol['orient_fix'] = fix      # cachear para que todos los cortes coincidan
    return fix

def extract_oriented_at(vol, idx, cfg, use_nlm=None):
    # Normaliza, toma el corte del plano correcto, corrige orientacion y (opcional) NLM
    data_norm = normalize_percentile(vol['data_raw'], cfg['pct_low'], cfg['pct_high'])
    plano = vol.get('plano', 'axial')
    sl = data_norm[idx, :, :] if plano == 'sagittal' else data_norm[:, :, idx]
    fix = _resolver_fix(vol, sl, cfg)
    sl_o = apply_orientation(sl, fix)
    if use_nlm is None: use_nlm = vol.get('nlm', False)
    if use_nlm:
        u8 = (np.clip(sl_o, 0, 1) * 255).astype(np.uint8)
        nlm = cv2.fastNlMeansDenoising(u8, None, h=12, templateWindowSize=7, searchWindowSize=21)
        sl_o = nlm.astype(np.float32) / 255.0
    return sl_o

def get_best_slice_oriented(vol, cfg):
    # Selecciona el mejor corte, lo orienta (ventriculos abajo) y corrige el spacing
    z_best, sl_raw, sp_2d = get_best_slice(vol, cfg)
    fix = _resolver_fix(vol, sl_raw, cfg)
    sl_o = apply_orientation(sl_raw, fix)
    if vol.get('nlm', False):
        u8 = (np.clip(sl_o, 0, 1) * 255).astype(np.uint8)
        nlm = cv2.fastNlMeansDenoising(u8, None, h=12, templateWindowSize=7, searchWindowSize=21)
        sl_o = nlm.astype(np.float32) / 255.0
    sp0, sp1 = sp_2d
    if int(fix[0]) % 2 == 1:        # rotacion 90/270 intercambia ejes en plano
        sp0, sp1 = sp1, sp0
    return z_best, sl_o, (sp0, sp1), fix

def segment_bilateral(slice_proc, brain_mask, cfg):
    # Segmentacion L/R unificada para CUALQUIER corte ya orientado.
    # Devuelve (lv, rv, midline). Escalera de respaldo evita mascaras vacias.
    mid, *_ = detect_midline(slice_proc, brain_mask)
    lv, rv = segment_ventricles(slice_proc, brain_mask, mid, cfg)
    if lv.sum() == 0 or rv.sum() == 0:
        # respaldo SUAVE: solo baja un poco el percentil y el tamaño minimo,
        # SIN aumentar los topes de area (para no reinflar la mascara).
        relajados = [
            {**cfg, 'ventricle_pct': max(80, cfg['ventricle_pct'] - 5),
                    'min_size': max(25, cfg['min_size'] // 2)},
            {**cfg, 'ventricle_pct': max(75, cfg['ventricle_pct'] - 9),
                    'min_size': max(20, cfg['min_size'] // 2), 'close_r': cfg['close_r'] + 1},
        ]
        for rc in relajados:
            l2, r2 = segment_ventricles(slice_proc, brain_mask, mid, rc)
            if l2.sum() > lv.sum(): lv = l2
            if r2.sum() > rv.sum(): rv = r2
            if lv.sum() > 0 and rv.sum() > 0: break
    return lv, rv, mid

def overlay_bilateral(ax, base, lv, rv, alpha=0.85):
    # Dibuja el corte en gris + ventriculo izq (AZUL solido) y der (ROJO solido).
    # Estilo "solido y plano" que reproduce la figura de referencia clinica:
    # un unico color azul para el ventriculo izquierdo y un unico rojo para el
    # derecho (en vez de un colormap con gradiente). Esto se logra construyendo
    # una imagen RGBA donde cada pixel del ventriculo recibe un color FIJO.
    ax.imshow(base.T, cmap='gray', origin='lower')
    H, W = base.shape
    # Color de referencia tomado de la figura objetivo:
    AZUL = (0.32, 0.55, 0.78)   # ventriculo izquierdo
    ROJO = (0.86, 0.36, 0.38)   # ventriculo derecho
    rgba = np.zeros((H, W, 4), dtype=float)
    if lv is not None and lv.sum() > 0:
        m = lv.astype(bool)
        rgba[m, 0], rgba[m, 1], rgba[m, 2], rgba[m, 3] = (*AZUL, alpha)
    if rv is not None and rv.sum() > 0:
        m = rv.astype(bool)
        rgba[m, 0], rgba[m, 1], rgba[m, 2], rgba[m, 3] = (*ROJO, alpha)
    # transpuesta para coincidir con la orientacion de imshow(.T, origin='lower')
    ax.imshow(np.transpose(rgba, (1, 0, 2)), origin='lower', interpolation='nearest')

def overlay_solido_unico(ax, base, mask, color=(0.86, 0.36, 0.38), alpha=0.85):
    # Variante mono-mascara (un solo color solido) para overlays de una sola
    # estructura (p. ej. plano sagital donde solo se segmenta un ventriculo).
    ax.imshow(base.T, cmap='gray', origin='lower')
    if mask is None or mask.sum() == 0:
        return
    H, W = base.shape
    rgba = np.zeros((H, W, 4), dtype=float)
    m = mask.astype(bool)
    rgba[m, 0], rgba[m, 1], rgba[m, 2], rgba[m, 3] = (*color, alpha)
    ax.imshow(np.transpose(rgba, (1, 0, 2)), origin='lower', interpolation='nearest')

def mosaico_resumen(volumenes, render_fn, titulo, fname, cols=5, panel=3.0, dpi=140):
    # Construye un mosaico (1 panel por volumen) y lo guarda como imagen de resumen
    n = len(volumenes)
    if n == 0:
        print('AVISO: no hay volúmenes cargados -> no se genera el mosaico "' + str(titulo) + '".')
        print('       Revisa la Sección A: probablemente las rutas de Google Drive no')
        print('       apuntan a tus archivos .nii (BASE_DIR / CONFIG["archivos"]).')
        return
    cols = min(cols, max(1, n)); rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * panel, rows * panel))
    fig.patch.set_facecolor('#0d1b2a'); axes = np.array(axes).reshape(-1)
    for i, vol in enumerate(volumenes):
        ax = axes[i]; ax.set_facecolor('#0d1b2a')
        try:
            render_fn(ax, vol)
        except Exception as e:
            ax.text(0.5, 0.5, f'{vol["id"]}\nERROR:\n{e}', ha='center', va='center',
                    color='red', fontsize=7, transform=ax.transAxes)
        ax.axis('off')
    for j in range(n, len(axes)): axes[j].axis('off')
    plt.suptitle(titulo, color='white', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTADOS_DIR, fname), dpi=dpi, facecolor='#0d1b2a')
    plt.show()

print('OK  Capa de orientacion + segmentacion bilateral cargada.')
print('    Funciones: apply_orientation, auto_ventricles_down, extract_oriented_at,')
print('               get_best_slice_oriented, segment_bilateral, overlay_bilateral, mosaico_resumen')


In [ ]:
# SECCIÓN C — Ejecución Etapa I para todos los volúmenes
print('Ejecutando Etapa I — Análisis Digital (orientado)...')
resultados_etapa1 = {}
for vol in volumenes:
    mid_idx = (vol['data_raw'].shape[0] if vol.get('plano')=='sagittal'
               else vol['data_raw'].shape[2]) // 2
    try:
        resultados_etapa1[vol['id']] = etapa_1_analisis_digital(vol, mid_idx)
    except Exception as e:
        print(f'ERROR en {vol["id"]}: {e}')

# ---- MOSAICO RESUMEN ETAPA I (cortes originales orientados) ----
def _render_e1(ax, vol):
    mid_idx = (vol['data_raw'].shape[0] if vol.get('plano')=='sagittal'
               else vol['data_raw'].shape[2]) // 2
    corte = extract_oriented_at(vol, mid_idx, CONFIG, use_nlm=False)
    ax.imshow(corte.T, cmap='gray', origin='lower',
              vmin=np.percentile(corte,0.5), vmax=np.percentile(corte,99.5))
    ax.set_title(f'{vol["id"]}\n{vol["tipo"]}', fontsize=8, color='white')
mosaico_resumen(volumenes, _render_e1,
    'RESUMEN ETAPA I — Base de datos MRI fetal (cortes originales orientados)',
    'RESUMEN_etapa1_database.png')
print('OK  Etapa I completada.')


---
## Sección E — Etapa II: Filtrado y Reducción de Ruido

**Tipo de ruido MRI: Riciano**

| Condición          | Distribución                             |
|--------------------|------------------------------------------|
| SNR alto (tejido)  | Riciana ≈ Gaussiana sesgada positivamente|
| Sin señal (fondo)  | Riciana → Rayleigh                       |

**Filtro seleccionado: GAUSSIANO** (kernel n=2 del profesor)
- Ruido Riciano es no impulsivo → suavizador lineal teóricamente adecuado
- Genera gradientes continuos favorables para umbralización del LCR
- La mediana está optimizada para ruido impulsivo (sal y pimienta)

La cadena v3 (CLAHE + Bilateral) se aplica adicionalmente para inhomogeneidad de campo.

---
**Defensa rápida (por qué Gaussiano y no Mediana):**
El ruido en magnitud de MRI sigue una distribución **Riciana** (no impulsiva). El filtro
**mediana** es óptimo para ruido *impulsivo* (sal y pimienta), que aquí no domina; aplicarlo
redondea las esquinas del atrio y desplaza el borde del LCR. El **Gaussiano** es un suavizador
lineal que preserva gradientes continuos, ideales para que la umbralización del LCR sea estable.
Se reporta **MSE** (magnitud del suavizado) y **preservación de bordes** (correlación de
gradientes Sobel) para justificar la elección de forma cuantitativa, no solo visual.


In [ ]:
def etapa_2_preprocesamiento(vol, z_idx, cfg):
    # Compara 3 filtros del profesor (sobre ROI 128x128) y aplica pipeline v3
    import time
    data_raw = vol['data_raw']; sx, sy, sz = vol['spacing']
    plano = vol.get('plano', 'axial')
    vol_norm = normalize_percentile(data_raw, cfg['pct_low'], cfg['pct_high'])
    corte_full = vol_norm[z_idx, :, :] if plano == 'sagittal' else vol_norm[:, :, z_idx]

    # ROI 128x128 centrado para funciones lentas del profesor
    cy = corte_full.shape[0]//2; cz = corte_full.shape[1]//2; hw=64
    yr0, yr1 = max(0,cy-hw), min(corte_full.shape[0],cy+hw)
    zr0, zr1 = max(0,cz-hw), min(corte_full.shape[1],cz+hw)
    roi = corte_full[yr0:yr1, zr0:zr1].astype(float)
    roi_u8 = (roi * 255).astype(int)

    snr_roi = compute_snr(roi)
    print(f'\n{vol["id"]}  (z={z_idx})  ROI={roi.shape}  SNR={snr_roi:.1f}')
    print('  Aplicando filtros del profesor sobre ROI 128x128 (puede tardar)...')

    t0=time.time(); f_gauss = filtro_pro(roi_u8, 2).astype(float)
    print(f'  Gaussiano: {time.time()-t0:.1f}s')
    t0=time.time(); f_median = filtro_mediana(roi_u8, cfg['dsp_median_tam']).astype(float)
    print(f'  Mediana:   {time.time()-t0:.1f}s')
    t0=time.time(); f_avg    = filtro_pro(roi_u8, 1).astype(float)
    print(f'  Promedio:  {time.time()-t0:.1f}s')

    mse_g=mse(roi_u8,f_gauss); mse_m=mse(roi_u8,f_median); mse_a=mse(roi_u8,f_avg)

    def corr_sobel(orig, filt):
        s_o,_,_=bordes_sobel(orig.astype(float)); s_f,_,_=bordes_sobel(filt.astype(float))
        if s_o.std()<1e-8 or s_f.std()<1e-8: return 0.0
        return float(np.corrcoef(s_o.flatten(), s_f.flatten())[0,1])

    pres_g=corr_sobel(roi_u8,f_gauss); pres_m=corr_sobel(roi_u8,f_median); pres_a=corr_sobel(roi_u8,f_avg)

    df_comp = pd.DataFrame({
        'Filtro':       ['Gaussiano (kernel 2)', 'Mediana', 'Promedio (kernel 1)'],
        'MSE':          [round(mse_g,2), round(mse_m,2), round(mse_a,2)],
        'Pres.Bordes':  [round(pres_g,3), round(pres_m,3), round(pres_a,3)],
        'Seleccionado': ['SI — Ruido Riciano', 'No — ruido impulsivo', 'No — destructivo'],
    })
    print('\n  Comparacion de filtros (sobre ROI 128x128):')
    display(df_comp)

    v3_result, snr_v3, sigma_v3 = preprocess_slice_v3(roi, cfg)
    f_g_norm  = (f_gauss-f_gauss.min())/(f_gauss.max()-f_gauss.min()+1e-8)
    roi_norm  = roi/(roi.max()+1e-8)
    residual  = np.abs(roi_norm - f_g_norm)

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    titles = [f'Original (z={z_idx})', f'Gaussiano (MSE={mse_g:.1f})',
              f'Mediana (MSE={mse_m:.1f})', f'Promedio (MSE={mse_a:.1f})',
              f'Pipeline v3 (SNR={snr_v3:.1f})', 'Residual de ruido']
    imgs = [roi_norm, f_g_norm,
            (f_median-f_median.min())/(f_median.max()-f_median.min()+1e-8),
            (f_avg-f_avg.min())/(f_avg.max()-f_avg.min()+1e-8),
            v3_result, residual]
    cmaps = ['gray','gray','gray','gray','gray','hot']
    for ax, img, title, cmap in zip(axes.flatten(), imgs, titles, cmaps):
        ax.imshow(img.T, cmap=cmap, origin='lower'); ax.set_title(title, fontsize=9); ax.axis('off')
    plt.suptitle(f'Etapa II — Filtrado · {vol["id"]}', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTADOS_DIR, f'E_etapa2_{vol["id"]}.png'), dpi=100)
    plt.show()
    return df_comp

In [ ]:
print('Ejecutando Etapa II — Filtrado para todos los volúmenes...')
print('NOTA: Funciones del profesor con bucles Python pueden tardar varios minutos por volumen.')
resultados_etapa2 = {}
for vol in volumenes:
    z_mid = (vol['data_raw'].shape[0] if vol.get('plano')=='sagittal'
             else vol['data_raw'].shape[2]) // 2
    try:
        df_e2 = etapa_2_preprocesamiento(vol, z_mid, CONFIG)
        resultados_etapa2[vol['id']] = df_e2
    except Exception as e:
        print(f'ERROR en {vol["id"]}: {e}')

if resultados_etapa2:
    print('\nTabla MSE cruzada resumen:')
    filas_mse = []
    for vid, df in resultados_etapa2.items():
        row = {'Volumen': vid}
        for _, r in df.iterrows():
            row[r['Filtro']] = r['MSE']
        filas_mse.append(row)
    df_mse = pd.DataFrame(filas_mse).set_index('Volumen')
    display(df_mse)
    df_mse.to_csv(os.path.join(RESULTADOS_DIR, 'E_mse_cruzada.csv'))
print('OK  Etapa II completada.')
# ---- MOSAICO RESUMEN ETAPA II (procesado v3 orientado) ----
def _render_e2(ax, vol):
    z_b, sl_ob, sp2, fx = get_best_slice_oriented(vol, CONFIG)
    spx, snr, sig = preprocess_slice_v3(sl_ob, CONFIG)
    ax.imshow(spx.T, cmap='gray', origin='lower')
    ax.set_title(f'{vol["id"]}\nSNR={snr:.1f} sig={sig}', fontsize=8, color='white')
mosaico_resumen(volumenes, _render_e2,
    'RESUMEN ETAPA II — Procesamiento y reducción de ruido (CLAHE+Bilateral+NLM)',
    'RESUMEN_etapa2_denoise.png')


---
## Sección F — Etapa III: Segmentación Ventricular

5 estrategias comparadas contra la segmentación v3 (referencia de calidad):

| # | Método | Base |
|---|--------|------|
| 1 | Umbral global (mu+1.5*sigma) + morfología profesor | Profesor |
| 2 | Umbral adaptativo (media local gaussiana) | Nueva |
| 3 | K-means (k=2, cluster mayor intensidad) | Profesor |
| 4 | Bordes Canny + relleno + intersección LCR | Nueva |
| 5 | Segmentación v3 (cisura + percentil 90) | **Referencia** |

In [ ]:
def dice_score(mask_a, mask_b):
    a = mask_a.astype(bool); b = mask_b.astype(bool)
    return 2*(a&b).sum() / (a.sum()+b.sum()+1e-8)

def etapa_3_segmentacion(slice_filt, slice_raw, spacing, nombre, cfg, plano='axial'):
    # 4 estrategias clasicas + estrategia v3 BILATERAL (L=azul / R=rojo) para TODOS
    sx, sy = spacing[0], spacing[1]
    thresh_brain = threshold_otsu(slice_filt)
    brain_mask   = remove_small_objects(slice_filt > thresh_brain, min_size=100)
    mu_t  = slice_filt[brain_mask].mean() if brain_mask.sum()>0 else 0.5
    sig_t = slice_filt[brain_mask].std()  if brain_mask.sum()>0 else 0.1

    T_global = mu_t + 1.5*sig_t
    mask1 = (slice_filt > T_global) & brain_mask
    se_c  = crear_se('disco', 2*cfg['close_r']+1); se_o = crear_se('disco', 2*cfg['open_r']+1)
    mask1 = eliminar_objetos_pequenos(closing(opening(mask1.astype(float),se_o),se_c), cfg['min_area_px']).astype(bool)

    sigma_local = max(slice_filt.shape)*0.1
    local_mean  = gaussian_filter(slice_filt, sigma=sigma_local)
    mask2 = (slice_filt > local_mean + 0.5*sig_t) & brain_mask
    mask2 = remove_small_objects(binary_opening(binary_closing(mask2,disk(cfg['close_r'])),disk(cfg['open_r'])), min_size=cfg['min_size'])

    mask3 = kmeans_simple(slice_filt, k=2, max_iter=20).astype(bool) & brain_mask
    mask3 = remove_small_objects(binary_closing(mask3, disk(cfg['close_r'])), min_size=cfg['min_size'])

    brain_ero = binary_opening(brain_mask, disk(5))
    edges     = cv2.Canny((slice_filt*255).astype(np.uint8), 20, 80)
    edges_cl  = binary_closing((edges>0) & brain_ero, disk(3))
    filled    = binary_fill_holes(edges_cl)
    mask4 = remove_small_objects(filled & (slice_filt > np.percentile(slice_filt[brain_mask],75)) & brain_mask, min_size=cfg['min_size']) if brain_mask.sum()>0 else np.zeros_like(brain_mask)

    # Estrategia 5 — v3 BILATERAL unificada (siempre intenta L y R)
    lv5, rv5, midline_col = segment_bilateral(slice_filt, brain_mask, cfg)
    mask5 = (lv5|rv5).astype(bool)

    masks = [mask1, mask2, mask3, mask4, mask5]
    nombres_s = ['Umbral global', 'Umbral adaptativo', 'K-means', 'Canny+relleno', 'v3 bilateral (ref)']
    px_mm2 = sx*sy
    datos = [{'Estrategia':nm,'Area_px':int(mk.sum()),'Area_mm2':round(mk.sum()*px_mm2,1),
              'Dice_vs_v3':round(dice_score(mk,mask5),3)} for nm,mk in zip(nombres_s,masks)]
    df_s = pd.DataFrame(datos)
    print(f'\n  {nombre} — Métricas de segmentación:'); display(df_s)

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes[0,0].imshow(slice_filt.T, cmap='gray', origin='lower'); axes[0,0].set_title('Imagen filtrada (v3)')
    for ax, mk, nm in zip(axes.flatten()[1:5], masks[:4], nombres_s[:4]):
        ax.imshow(slice_filt.T, cmap='gray', origin='lower', alpha=0.7)
        ax.imshow(np.ma.masked_where(~mk.T,mk.T), cmap='Reds', alpha=0.6, origin='lower')
        ax.set_title(f'{nm}\nArea={int(mk.sum())}px  Dice={dice_score(mk,mask5):.3f}', fontsize=9); ax.axis('off')
    ax5 = axes[1,2]
    overlay_bilateral(ax5, slice_filt, lv5, rv5, alpha=0.75)
    ax5.axvline(midline_col, color='cyan', lw=1.5, ls='--')
    ax5.set_title('v3 bilateral  (L=azul  R=rojo  cisura=cian)', fontsize=9); ax5.axis('off')
    axes[0,0].axis('off')
    plt.suptitle(f'Etapa III — Segmentación Ventricular · {nombre}', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTADOS_DIR, f'F_etapa3_{nombre}.png'), dpi=100)
    plt.show()
    return mask5, lv5, rv5, midline_col, df_s

def consistencia_intercorte(vol, z_best, cfg, ventana=3):
    # Segmentacion v3 BILATERAL en z_best±ventana -> area, Dice inter-corte y CV (orientado)
    sx, sy, sz = vol['spacing']; plano = vol.get('plano','axial')
    n_ax  = vol['data_raw'].shape[0] if plano == 'sagittal' else vol['data_raw'].shape[2]
    # spacing 2D coherente con la orientacion
    fix = vol.get('orient_fix') or (0, False, False)
    sp_2d = (sy, sz) if plano == 'sagittal' else (sx, sy)
    if int(fix[0]) % 2 == 1: sp_2d = (sp_2d[1], sp_2d[0])
    resultados_mp = []
    for dz in range(-ventana, ventana+1):
        z_c = z_best + dz
        if not (0 <= z_c < n_ax): continue
        try:
            sl_c = extract_oriented_at(vol, z_c, cfg)
            sp_c, _, _ = preprocess_slice_v3(sl_c, cfg)
            bm_c = remove_small_objects(sp_c > threshold_otsu(sp_c), min_size=100)
            lc, rc, _ = segment_bilateral(sp_c, bm_c, cfg)
            wlc, _ = min_bounding_rect_width(lc, sp_2d[0], sp_2d[1])
            wrc, _ = min_bounding_rect_width(rc, sp_2d[0], sp_2d[1])
            w_c    = max(wlc, wrc)
            if w_c > 0: resultados_mp.append({'z':z_c,'w_mm':w_c,'mask':(lc|rc)})
        except Exception: pass
    if len(resultados_mp) < 2:
        return (resultados_mp[0]['w_mm'] if resultados_mp else 0.0), 0.0
    ws = [r['w_mm'] for r in resultados_mp]; zs = [r['z'] for r in resultados_mp]
    mean_w = float(np.mean(ws)); cv = float(np.std(ws)/(mean_w+1e-8))
    dices_c = []
    for i in range(len(resultados_mp)-1):
        m_a = resultados_mp[i]['mask']; m_b = resultados_mp[i+1]['mask']
        if m_a.shape == m_b.shape: dices_c.append(dice_score(m_a,m_b))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    col_w = ['limegreen' if w<10 else 'gold' if w<12.5 else 'orange' if w<15 else 'red' for w in ws]
    ax1.bar(range(len(zs)), ws, color=col_w)
    ax1.set_xticks(range(len(zs))); ax1.set_xticklabels([f'z={z}' for z in zs], fontsize=8)
    for th_val,th_col in [(cfg['mild_mm'],'gold'),(cfg['moderate_mm'],'orange'),(cfg['severe_mm'],'red')]:
        ax1.axhline(th_val, color=th_col, lw=1, ls='--')
    ax1.axhline(mean_w, color='black', lw=2, label=f'Media={mean_w:.1f}mm')
    ax1.set_ylabel('Diámetro (mm)'); ax1.set_title(f'Consistencia · {vol["id"]}\nCV={cv*100:.1f}%'); ax1.legend(fontsize=7)
    if dices_c:
        ax2.bar(range(len(dices_c)), dices_c, color='steelblue'); ax2.set_ylim(0,1)
        ax2.set_ylabel('Dice inter-corte'); ax2.set_title(f'Dice consecutivos (media={np.mean(dices_c):.3f})')
    plt.suptitle(f'Consistencia inter-corte · {vol["id"]}', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTADOS_DIR, f'F_consistencia_{vol["id"]}.png'), dpi=100)
    plt.show()
    print(f'  {vol["id"]}: Media={mean_w:.2f}mm  CV={cv*100:.1f}%  '
          f'{"Consistente" if cv<cfg["cv_thresh"] else "Revisar"}')
    return mean_w if cv<cfg['cv_thresh'] else ws[len(ws)//2], cv


In [ ]:
print('Ejecutando Etapa III — Segmentación (orientada + bilateral) para todos los volúmenes...')
resultados_etapa3 = {}
filas_resumen_seg = []

for vol in volumenes:
    try:
        z_best, sl_o, sp_2d, fix = get_best_slice_oriented(vol, CONFIG)
        sl_proc, snr_v, sig_v = preprocess_slice_v3(sl_o, CONFIG)
        mask5, lv5, rv5, midline, df_s = etapa_3_segmentacion(
            sl_proc, sl_o, sp_2d, vol['id'], CONFIG, plano=vol.get('plano','axial'))
        w_final, cv_val = consistencia_intercorte(vol, z_best, CONFIG, ventana=CONFIG['cross_slices'])

        wl, _ = min_bounding_rect_width(lv5, sp_2d[0], sp_2d[1])
        wr, _ = min_bounding_rect_width(rv5, sp_2d[0], sp_2d[1])
        w_diag = w_final if w_final and w_final > 0 else max(wl, wr)
        clasif, col_cl = diagnose(w_diag, CONFIG)
        found_L = int(lv5.sum() > 0); found_R = int(rv5.sum() > 0)

        print(f'\n  {vol["id"]} ({vol["tipo"]})  z={z_best}  fix={fix}  SNR={snr_v:.1f}')
        print(f'  ├─ Izq: {"OK" if found_L else "NO":3s} px={int(lv5.sum())} D={wl:.2f}mm   '
              f'Der: {"OK" if found_R else "NO":3s} px={int(rv5.sum())} D={wr:.2f}mm')
        print(f'  └─ D final={w_diag:.2f}mm  CV={cv_val*100:.1f}%  -> {clasif}')

        filas_resumen_seg.append({'Volumen':vol['id'],'Tipo':vol['tipo'],'z_best':z_best,
            'SNR':round(snr_v,1),'Vent_Izq':'SI' if found_L else 'NO','Px_Izq':int(lv5.sum()),
            'D_Izq_mm':round(wl,2),'Vent_Der':'SI' if found_R else 'NO','Px_Der':int(rv5.sum()),
            'D_Der_mm':round(wr,2),'Total_Vent':found_L+found_R,'D_final_mm':round(w_diag,2),
            'CV_pct':round(cv_val*100,1),'Diagnostico':clasif})

        resultados_etapa3[vol['id']] = {'z_best':z_best,'snr':snr_v,'mask_v3':mask5,
            'lv':lv5,'rv':rv5,'midline':midline,'sl_proc':sl_proc,'w_final':w_final,
            'cv':cv_val,'df_strat':df_s,'spacing':sp_2d,'plano':vol.get('plano','axial')}
    except Exception as e:
        print(f'ERROR en {vol["id"]}: {e}'); import traceback; traceback.print_exc()

print(f'\n{"="*65}\n  Etapa III completada: {len(resultados_etapa3)} volúmenes.\n{"="*65}')
if filas_resumen_seg:
    df_seg_resumen = pd.DataFrame(filas_resumen_seg)
    display(df_seg_resumen)
    df_seg_resumen.to_csv(os.path.join(RESULTADOS_DIR, 'F_resumen_segmentacion.csv'), index=False)
    n_ambos = int((df_seg_resumen['Total_Vent']==2).sum())
    print(f'\n  Ambos ventrículos detectados: {n_ambos}/{len(filas_resumen_seg)}')

# ---- MOSAICO RESUMEN ETAPA III (blue/red para todos) ----
def _render_seg(ax, vol):
    r = resultados_etapa3.get(vol['id'])
    if r is None: raise ValueError('sin segmentacion')
    overlay_bilateral(ax, r['sl_proc'], r['lv'], r['rv'])
    ax.axvline(r['midline'], color='cyan', lw=1.0, ls='--')
    cl, _ = diagnose(r['w_final'] if r['w_final'] else 0, CONFIG)
    ax.set_title(f'{vol["id"]}\nL={int(r["lv"].sum())}px R={int(r["rv"].sum())}px',
                 fontsize=8, color='white')
if resultados_etapa3:
    mosaico_resumen(volumenes, _render_seg,
        'RESUMEN ETAPA III — Segmentación ventricular (L=azul · R=rojo · cisura=cian)',
        'RESUMEN_etapa3_segmentacion.png')


---
## Sección G — Etapa IV-V: Cuantificación y Clasificación

Calcula el **diámetro atrial** de ambos ventrículos usando el rectángulo mínimo
rotado (`cv2.minAreaRect`), que equivale al eje menor del ventrículo lateral.

| Métrica | Unidad |
|---|---|
| Diámetro atrial (eje menor del rect. mínimo) | mm |
| Eje mayor | mm | Área | mm² |
| Excentricidad (elipse ajustada) | — |
| Compacidad = 4pi·Area/Perimetro² | — |

**Validación multiplanar:** CV < 0.25 → usar media; si no → usar corte único y advertir.

In [ ]:
def etapa_4_5_cuantificacion(mask_L, mask_R, slice_proc, spacing, nombre, cfg, w_final=None, cv_val=None):
    # Cuantificación geométrica + clasificación diagnóstica
    # mask_L, mask_R: máscaras booleanas 2D de ventrículos izq/der
    sx, sy, sz = spacing; px_to_mm2 = sx*sy

    def metricas_ventriculo(mask, lado):
        if mask.sum() == 0:
            return {'lado':lado,'diametro_mm':0,'eje_mayor_mm':0,'eje_menor_mm':0,
                    'area_mm2':0,'excentricidad':0,'compacidad':0,'relacion_ejes':0,
                    'rect':None,'ellipse':None}
        w_mm, rect = min_bounding_rect_width(mask, sx, sy)
        area_mm2   = float(mask.sum()*px_to_mm2)
        labeled, _ = scipy_label(mask)
        props = measure.regionprops(labeled)
        if props:
            p = max(props, key=lambda x: x.area)
            excentricidad = p.eccentricity
            compacidad    = 4*np.pi*p.area/(p.perimeter**2+1e-8) if p.perimeter>0 else 0
            eje_mayor_mm  = p.major_axis_length*np.mean([sx, sy])
            eje_menor_mm  = p.minor_axis_length*np.mean([sx, sy])
        else:
            excentricidad=eje_mayor_mm=eje_menor_mm=compacidad=0
        img_u8      = (mask.astype(np.uint8))*255
        contours, _ = cv2.findContours(img_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        ellipse = None
        if contours:
            largest = max(contours, key=cv2.contourArea)
            if len(largest) >= 5:
                try: ellipse = cv2.fitEllipse(largest)
                except Exception: pass
        return {'lado':lado,'diametro_mm':round(w_mm,2),'eje_mayor_mm':round(eje_mayor_mm,2),
                'eje_menor_mm':round(eje_menor_mm,2),'area_mm2':round(area_mm2,2),
                'excentricidad':round(excentricidad,3),'compacidad':round(compacidad,3),
                'relacion_ejes':round(eje_mayor_mm/(eje_menor_mm+1e-8),2),'rect':rect,'ellipse':ellipse}

    m_L = metricas_ventriculo(mask_L, 'Izquierdo')
    m_R = metricas_ventriculo(mask_R, 'Derecho')
    diam_final = w_final if w_final and w_final>0 else max(m_L['diametro_mm'], m_R['diametro_mm'])
    clasif, color_clasif = diagnose(diam_final, cfg)

    campos  = ['diametro_mm','eje_mayor_mm','eje_menor_mm','area_mm2','excentricidad','compacidad','relacion_ejes']
    nc      = ['Diámetro atrial (mm)','Eje mayor (mm)','Eje menor (mm)','Área (mm²)','Excentricidad','Compacidad','Relacion ejes']
    df_met  = pd.DataFrame([{'Métrica':n,'Izquierdo':m_L[c],'Derecho':m_R[c]} for c,n in zip(campos,nc)])
    display(df_met)
    print(f'\n  {nombre}  —  Diámetro final: {diam_final:.2f} mm  ->  {clasif}')
    if cv_val is not None:
        print(f'  Validación multiplanar: CV={cv_val*100:.1f}%  '
              f'({"Consistente" if cv_val<cfg["cv_thresh"] else "Revisar"})')

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))

    # 1. Overlay ambas máscaras — estilo SOLIDO (azul/rojo planos, como la figura clinica)
    overlay_bilateral(axes[0,0], slice_proc, mask_L, mask_R, alpha=0.85)
    axes[0,0].set_title('Ambos ventrículos (Izq=azul, Der=rojo)'); axes[0,0].axis('off')

    # 2. Ventrículo izquierdo con rect mínimo
    axes[0,1].imshow(slice_proc.T, cmap='gray', origin='lower', alpha=0.7)
    if mask_L.sum()>0: axes[0,1].imshow(np.ma.masked_where(~mask_L.T,mask_L.T),cmap='Blues',alpha=0.6,origin='lower')
    draw_rect_on_ax(axes[0,1], m_L['rect'], 'deepskyblue', f'L={m_L["diametro_mm"]:.1f}mm')
    axes[0,1].set_title(f'Izquierdo  D={m_L["diametro_mm"]:.2f}mm\nArea={m_L["area_mm2"]:.1f}mm2')
    axes[0,1].legend(fontsize=7); axes[0,1].axis('off')

    # 3. Ventrículo derecho con rect mínimo
    axes[0,2].imshow(slice_proc.T, cmap='gray', origin='lower', alpha=0.7)
    if mask_R.sum()>0: axes[0,2].imshow(np.ma.masked_where(~mask_R.T,mask_R.T),cmap='Reds',alpha=0.6,origin='lower')
    draw_rect_on_ax(axes[0,2], m_R['rect'], 'tomato', f'R={m_R["diametro_mm"]:.1f}mm')
    axes[0,2].set_title(f'Derecho  D={m_R["diametro_mm"]:.2f}mm\nArea={m_R["area_mm2"]:.1f}mm2')
    axes[0,2].legend(fontsize=7); axes[0,2].axis('off')

    # 4. Tabla como figura
    axes[1,0].axis('off')
    tabla_data = [[r['Métrica'],str(r['Izquierdo']),str(r['Derecho'])] for _,r in df_met.iterrows()]
    tabla = axes[1,0].table(cellText=tabla_data, colLabels=['Métrica','Izq','Der'],
                             loc='center', cellLoc='left')
    tabla.auto_set_font_size(False); tabla.set_fontsize(8); tabla.scale(1.2,1.3)
    axes[1,0].set_title('Tabla de métricas', pad=20)

    # 5. Escala clínica
    axes[1,1].set_xlim(0, 25); axes[1,1].set_ylim(-0.5, 1.5)
    for thr,col,etq in [(0,'limegreen','Normal'),(cfg['mild_mm'],'gold','Leve'),
                         (cfg['moderate_mm'],'orange','Moderada'),(cfg['severe_mm'],'red','Severa')]:
        next_thr = {'Normal':cfg['mild_mm'],'Leve':cfg['moderate_mm'],'Moderada':cfg['severe_mm'],'Severa':25}[etq]
        axes[1,1].barh(0, next_thr-thr, left=thr, height=0.4, color=col, alpha=0.7, label=etq)
    axes[1,1].axvline(diam_final, color='white', lw=3, zorder=5)
    axes[1,1].text(diam_final, 0.7, f'{diam_final:.1f}mm', ha='center', fontsize=10, color='white', fontweight='bold')
    axes[1,1].set_xlabel('Diámetro atrial (mm)'); axes[1,1].set_title('Escala clínica')
    axes[1,1].legend(loc='upper right', fontsize=7); axes[1,1].set_yticks([])

    # 6. Diagnóstico final
    axes[1,2].axis('off')
    cv_texto = f'CV multiplanar: {cv_val*100:.1f}%\n{"Consistente" if cv_val<cfg["cv_thresh"] else "Revisar"}' if cv_val else ''
    axes[1,2].text(0.5, 0.5,
        f'DIAGNÓSTICO FINAL\n\nArchivo: {nombre}\n\n'
        f'Diámetro L: {m_L["diametro_mm"]:.2f} mm\n'
        f'Diámetro R: {m_R["diametro_mm"]:.2f} mm\n'
        f'Diámetro final: {diam_final:.2f} mm\n\n'
        f'Clasificación:\n{clasif}\n\n{cv_texto}',
        transform=axes[1,2].transAxes, fontsize=11, va='center', ha='center',
        bbox=dict(boxstyle='round,pad=0.6', facecolor=color_clasif, alpha=0.3))

    plt.suptitle(f'Etapa IV-V — Cuantificación · {nombre}  ->  {clasif} ({diam_final:.1f}mm)',
                 fontsize=12, color=color_clasif)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTADOS_DIR, f'G_etapa45_{nombre}.png'), dpi=100)
    plt.show()
    return {'nombre':nombre,'diam_L':m_L['diametro_mm'],'diam_R':m_R['diametro_mm'],
            'diam_final':diam_final,'clasificacion':clasif,'color':color_clasif}

In [ ]:
print('Ejecutando Etapa IV-V — Cuantificación y Clasificación...')
resultados_etapa45 = {}
for vol in volumenes:
    if vol['id'] not in resultados_etapa3:
        print(f'AVISO: {vol["id"]} sin resultados de Etapa III. Saltando.')
        continue
    r3 = resultados_etapa3[vol['id']]
    try:
        res = etapa_4_5_cuantificacion(
            r3['lv'], r3['rv'], r3['sl_proc'], r3['spacing'], vol['id'], CONFIG,
            w_final=r3.get('w_final'), cv_val=r3.get('cv'))
        resultados_etapa45[vol['id']] = res
    except Exception as e:
        print(f'ERROR en {vol["id"]}: {e}')

print(f'\n{"="*65}')
print(f'  Etapa IV-V completada: {len(resultados_etapa45)} volúmenes.')
print(f'{"="*65}')
print(f'  {"Volumen":<20} {"Tipo":<14} {"D_Izq":>7} {"D_Der":>7} {"D_Final":>8}  Clasificacion')
print(f'  {"-"*20} {"-"*14} {"-"*7} {"-"*7} {"-"*8}  {"-"*14}')
for vid, res in resultados_etapa45.items():
    vt = next((v["tipo"] for v in volumenes if v["id"]==vid), "—")
    print(f'  {vid:<20} {vt:<14} {res["diam_L"]:>6.2f}mm {res["diam_R"]:>6.2f}mm {res["diam_final"]:>7.2f}mm  {res["clasificacion"]}')

---
## Sección H — CNN con Transfer Learning: Attention U-Net + EfficientNet-B0

### Referencias Bibliográficas — Arquitectura CNN

**Arquitectura principal:**
> Oktay O. et al., "Attention U-Net: Learning Where to Look for the Pancreas,"
> *MIDL 2018.* arXiv:1804.03999

**Encoder backbone:**
> Tan M. & Le Q.V., "EfficientNet: Rethinking Model Scaling for CNNs,"
> *ICML 2019.* arXiv:1905.11946. Pesos: ImageNet ILSVRC 2012.

**Transfer learning en imágenes médicas:**
> Raghu M. et al., "Transfusion: Understanding Transfer Learning for Medical Imaging,"
> *NeurIPS 2019.* arXiv:1902.07208
> *"Características de ImageNet transfieren efectivamente a tareas médicas con datos limitados."*

**Base de datos fetal (referencia):**
> Payette K. et al., "An automatic multi-tissue human fetal brain segmentation benchmark
> using the Fetal Tissue Annotation Dataset (FeTA)," *Scientific Data* 8, 330 (2021).

**Función de pérdida Dice:**
> Milletari F. et al., "V-Net: Fully Convolutional Neural Networks for Volumetric
> Medical Image Segmentation," *3DV 2016.* arXiv:1606.04797

### Justificación
- **Attention U-Net:** gates focalizan en el ventrículo pequeño, ignorando corteza irrelevante
- **EfficientNet-B0:** ~5M params vs ~19M ResNet-50 — apto para CPU
- **Sin ground truth:** pseudo-etiquetas de v3 (estrategia validada en Bai et al., MICCAI 2017)

In [ ]:
# SECCIÓN H — CNN con Transfer Learning

def generar_pseudo_etiqueta(vol, z_idx, cfg):
    # Genera máscara pseudo-GT para un corte usando pipeline v3
    # Combina ventrículo izquierdo + derecho en máscara binaria única
    try:
        plano = vol.get('plano', 'axial')
        data_norm = normalize_percentile(vol['data_raw'], cfg['pct_low'], cfg['pct_high'])
        sl_raw    = data_norm[z_idx,:,:] if plano=='sagittal' else data_norm[:,:,z_idx]
        sl_proc, _, _ = preprocess_slice_v3(sl_raw, cfg)
        t_b = threshold_otsu(sl_proc)
        bm  = remove_small_objects(sl_proc > t_b, min_size=100)
        if plano == 'sagittal':
            vent = segment_sagittal_ventricle(sl_proc, bm, cfg)
            return vent.astype(np.uint8), sl_proc
        else:
            mid, *_ = detect_midline(sl_proc, bm)
            lv, rv  = segment_ventricles(sl_proc, bm, mid, cfg)
            return (lv|rv).astype(np.uint8), sl_proc
    except Exception:
        h, w = vol['data_raw'].shape[:2]
        return np.zeros((h,w), dtype=np.uint8), np.zeros((h,w))

print('OK  Funcion generar_pseudo_etiqueta definida.')

In [ ]:
class DatasetVentriculos(Dataset):
    # Dataset PyTorch para segmentación ventricular fetal
    # Imagen: corte preprocesado v3 redimensionado a target_size
    # Etiqueta: pseudo-máscara generada por pipeline v3
    # Solo incluye cortes con ventrículo detectable (mask.sum() > min_area_px)
    def __init__(self, volumenes, cfg, augment=True):
        self.muestras = []
        self.augment  = augment
        self.cfg      = cfg
        target_h, target_w = cfg['target_size']

        if ALB_DISPONIBLE and augment:
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.Rotate(limit=15, p=0.5),
                A.RandomBrightnessContrast(p=0.3),
                A.Resize(target_h, target_w),
                A.Normalize(mean=[0.5], std=[0.5]),
                ToTensorV2(),
            ])
        else:
            self.transform = None

        print('Generando pseudo-etiquetas para el dataset CNN...')
        for vol in volumenes:
            plano_v = vol.get('plano', 'axial')
            n_slices = vol['data_raw'].shape[0] if plano_v=='sagittal' else vol['data_raw'].shape[2]
            z_min = int(n_slices*cfg['z_range_pct'][0]); z_max = int(n_slices*cfg['z_range_pct'][1])
            for z_idx in range(z_min, z_max):
                mascara, sl_proc = generar_pseudo_etiqueta(vol, z_idx, cfg)
                if mascara.sum() > cfg['min_area_px']:
                    img_r  = cv2.resize(sl_proc.astype(np.float32), (target_w, target_h), interpolation=cv2.INTER_LINEAR)
                    mask_r = cv2.resize(mascara.astype(np.float32), (target_w, target_h), interpolation=cv2.INTER_NEAREST)
                    self.muestras.append((img_r, (mask_r>0.5).astype(np.float32)))
        print(f'OK  Dataset: {len(self.muestras)} muestras válidas.')

    def __len__(self): return len(self.muestras)

    def __getitem__(self, idx):
        img, mask = self.muestras[idx]
        if self.transform is not None and self.augment and ALB_DISPONIBLE:
            img_u8 = (img*255).astype(np.uint8)
            result = self.transform(image=img_u8, mask=mask)
            return result['image'].float(), result['mask'].unsqueeze(0).float()
        else:
            img_t  = (torch.tensor(img).unsqueeze(0).float() - 0.5) / 0.5
            mask_t = torch.tensor(mask).unsqueeze(0).float()
            return img_t, mask_t

print('OK  Clase DatasetVentriculos definida.')

In [ ]:
def crear_modelo_cnn(cfg):
    # ==================================================================
    # ARQUITECTURA: U-Net con compuertas de atencion (decoder_attention_type='scse')
    #               + encoder EfficientNet-B0 preentrenado en ImageNet.
    #
    # POR QUE CNN Y NO OTRO CLASIFICADOR (SVM / RandomForest / umbral):
    #   La segmentacion es un problema pixel-a-pixel con CONTEXTO ESPACIAL.
    #   Un SVM/RF sobre intensidades pierde la forma; una CNN aprende textura +
    #   forma + posicion del ventriculo a la vez (campos receptivos jerarquicos).
    #
    # POR QUE U-NET: es el estandar en segmentacion biomedica. Las "skip
    #   connections" recuperan detalle fino (bordes del atrio) que el encoder
    #   comprime. Ref: Ronneberger 2015 / Milletari (V-Net) 2016.
    #
    # POR QUE ATENCION (Attention U-Net, Oktay 2018): las attention gates dejan
    #   pasar solo las regiones relevantes (el ventriculo, pequeño) y suprimen
    #   corteza/fondo irrelevante -> mejor Dice con pocos datos.
    #
    # POR QUE EfficientNet-B0 COMO ENCODER: ~5.3M parametros (vs ~25M de ResNet-50)
    #   con accuracy ImageNet comparable; cabe en CPU/Colab y reduce sobreajuste
    #   en un dataset chico (10 volumenes). Ref: Tan & Le, ICML 2019.
    #
    # POR QUE TRANSFER LEARNING (encoder ImageNet): con N pequeño no hay datos
    #   para aprender bordes/texturas desde cero. Las capas tempranas de ImageNet
    #   ya son detectores genericos que transfieren a MRI. Ref: Raghu 2019.
    #
    # Entrada: 1 canal (MRI gris); Salida: 1 canal (logit de prob. ventriculo).
    # ==================================================================
    if not SMP_DISPONIBLE:
        print('AVISO: smp no disponible. Modelo no creado.'); return None
    modelo = smp.Unet(
        encoder_name         = cfg['encoder_name'],
        encoder_weights      = 'imagenet',
        in_channels          = 1,
        classes              = 1,
        activation           = None,
        decoder_attention_type = 'scse',
    )
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    return modelo.to(device)


class PerdidaDiceBCE(nn.Module):
    # Perdida combinada: L = 0.5*DiceLoss + 0.5*BCEWithLogitsLoss
    #
    # DICE vs IoU (pregunta del jurado):
    #   Dice = 2|P n G| / (|P| + |G|)         (= F1 sobre pixeles)
    #   IoU  =  |P n G| / |P u G|             (Jaccard)
    #   Relacion: IoU = Dice / (2 - Dice). Ambos miden solapamiento mascara-vs-GT,
    #   pero Dice pondera mas la interseccion (mas "indulgente") y es mejor senal
    #   de gradiente para entrenar con clases desbalanceadas (el ventriculo es
    #   diminuto vs el fondo). Por eso ENTRENAMOS con Dice y REPORTAMOS ambos.
    #
    # POR QUE Dice + BCE (y no solo uno):
    #   - BCE da gradiente estable pixel-a-pixel pero se "ahoga" con el desbalance
    #     (predecir todo fondo ya da BCE baja).
    #   - Dice ataca directo el solapamiento de la clase rara (ventriculo).
    #   La suma 0.5/0.5 combina estabilidad (BCE) + foco en la clase rara (Dice).
    # Ref: Milletari et al., V-Net, 3DV 2016.
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.bce = nn.BCEWithLogitsLoss()

    def dice_loss(self, pred, target):
        pred_sig = torch.sigmoid(pred)
        inter    = (pred_sig*target).sum(dim=(2,3))
        denom    = pred_sig.sum(dim=(2,3)) + target.sum(dim=(2,3))
        return (1 - (2*inter+self.eps)/(denom+self.eps)).mean()

    def forward(self, pred, target):
        return 0.5*self.dice_loss(pred, target) + 0.5*self.bce(pred, target)

print('OK  Modelo y función de pérdida definidos.')

In [ ]:
def entrenar_modelo(modelo, train_loader, val_loader, cfg):
    # Entrenamiento en 2 fases: FASE 1 decoder (encoder congelado) / FASE 2 fine-tuning completo
    if modelo is None:
        print('AVISO: Modelo no disponible. Saltando.'); return [], [], []
    device   = next(modelo.parameters()).device
    criterio = PerdidaDiceBCE()
    n_epocas = cfg['cnn_epochs']; fase1_ep = n_epocas//2; fase2_ep = n_epocas-fase1_ep
    hist_train = []; hist_val = []; hist_dice = []
    mejor_dice = -1.0; mejor_estado = None

    def _run_epoch(loader, train=True, opt=None):
        if train: modelo.train()
        else:     modelo.eval()
        total_loss=0.0; total_dice=0.0; n=0
        ctx = torch.no_grad() if not train else torch.enable_grad()
        with ctx:
            for imgs, masks in loader:
                imgs, masks = imgs.to(device), masks.to(device)
                preds = modelo(imgs); loss = criterio(preds, masks)
                if train: opt.zero_grad(); loss.backward(); opt.step()
                pred_b = (torch.sigmoid(preds)>0.5).float()
                inter  = (pred_b*masks).sum().item(); denom = pred_b.sum().item()+masks.sum().item()
                total_dice += 2*inter/(denom+1e-6)
                total_loss += loss.item(); n+=1
        return total_loss/max(n,1), total_dice/max(n,1)

    # Fase 1: solo decoder
    print('\n--- FASE 1: Decoder solamente (encoder congelado) ---')
    for p in modelo.encoder.parameters(): p.requires_grad=False
    opt1 = torch.optim.Adam(filter(lambda p: p.requires_grad, modelo.parameters()), lr=cfg['cnn_lr'])
    for ep in range(fase1_ep):
        tl, _ = _run_epoch(train_loader, True, opt1); vl, vd = _run_epoch(val_loader, False)
        hist_train.append(tl); hist_val.append(vl); hist_dice.append(vd)
        print(f'  F1 Ep{ep+1}/{fase1_ep}  train={tl:.4f}  val={vl:.4f}  dice={vd:.4f}')
        if vd>mejor_dice: mejor_dice=vd; mejor_estado={k:v.cpu().clone() for k,v in modelo.state_dict().items()}

    # Fase 2: fine-tuning completo
    print('\n--- FASE 2: Fine-tuning completo (encoder descongelado, LR/10) ---')
    for p in modelo.encoder.parameters(): p.requires_grad=True
    opt2  = torch.optim.Adam(modelo.parameters(), lr=cfg['cnn_lr']/10)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=fase2_ep)
    for ep in range(fase2_ep):
        tl, _ = _run_epoch(train_loader, True, opt2); vl, vd = _run_epoch(val_loader, False)
        sched.step()
        hist_train.append(tl); hist_val.append(vl); hist_dice.append(vd)
        print(f'  F2 Ep{ep+1}/{fase2_ep}  train={tl:.4f}  val={vl:.4f}  dice={vd:.4f}')
        if vd>mejor_dice: mejor_dice=vd; mejor_estado={k:v.cpu().clone() for k,v in modelo.state_dict().items()}

    if mejor_estado:
        modelo.load_state_dict({k:v.to(device) for k,v in mejor_estado.items()})
        print(f'\nMejor Dice de validación: {mejor_dice:.4f}')
    return hist_train, hist_val, hist_dice

print('OK  Función entrenar_modelo definida.')

In [ ]:
# Crear dataset, entrenar modelo CNN y graficar curvas de aprendizaje
modelo_cnn = None
hist_train = hist_val = hist_dice = []

if not SMP_DISPONIBLE:
    print('AVISO: SMP no disponible — saltando entrenamiento CNN.')
else:
    print('Creando dataset de pseudo-etiquetas...')
    dataset_full = DatasetVentriculos(volumenes, CONFIG, augment=True)

    if len(dataset_full) < 2:
        print('AVISO: Dataset < 2 muestras. Ajustar z_range_pct o min_area_px.')
    else:
        n_val   = max(1, int(len(dataset_full)*0.2))
        n_train = len(dataset_full)-n_val
        ds_train, ds_val = torch.utils.data.random_split(
            dataset_full, [n_train, n_val], generator=torch.Generator().manual_seed(42))
        train_loader = DataLoader(ds_train, batch_size=CONFIG['batch_size'], shuffle=True,  num_workers=0)
        val_loader   = DataLoader(ds_val,   batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)
        print(f'Train: {n_train}  |  Val: {n_val}  muestras')

        modelo_cnn = crear_modelo_cnn(CONFIG)
        if modelo_cnn is not None:
            device   = next(modelo_cnn.parameters()).device
            n_params = sum(p.numel() for p in modelo_cnn.parameters())
            print(f'Modelo en: {device}  |  Parámetros: {n_params:,}')
            hist_train, hist_val, hist_dice = entrenar_modelo(modelo_cnn, train_loader, val_loader, CONFIG)

            if hist_train:
                fase1_ep = CONFIG['cnn_epochs']//2
                fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
                ep_ax = range(1, len(hist_train)+1)
                ax1.plot(ep_ax, hist_train, 'b-o', ms=4, label='Train loss')
                ax1.plot(ep_ax, hist_val,   'r-o', ms=4, label='Val loss')
                ax1.axvline(fase1_ep+0.5, color='gray', ls='--', lw=1.5, label=f'Fase1->2 (ep{fase1_ep})')
                ax1.set_xlabel('Época'); ax1.set_ylabel('Pérdida (Dice+BCE)')
                ax1.set_title('Curva de pérdida — Entrenamiento CNN'); ax1.legend(); ax1.grid(alpha=0.3)
                ax2.plot(ep_ax, hist_dice, 'g-o', ms=4, label='Val Dice')
                ax2.axvline(fase1_ep+0.5, color='gray', ls='--', lw=1.5)
                best_ep = int(np.argmax(hist_dice))+1; best_d = max(hist_dice)
                ax2.axvline(best_ep, color='gold', ls=':', lw=2, label=f'Mejor Dice={best_d:.4f} (ep{best_ep})')
                ax2.set_xlabel('Época'); ax2.set_ylabel('Dice'); ax2.set_title('Curva Dice — Validación')
                ax2.set_ylim(0,1); ax2.legend(); ax2.grid(alpha=0.3)
                plt.suptitle('Sección H — Curvas de Entrenamiento CNN', fontsize=12)
                plt.tight_layout()
                plt.savefig(os.path.join(RESULTADOS_DIR, 'H_curvas_entrenamiento.png'), dpi=100)
                plt.show()

In [ ]:
def inferir_cnn(modelo, vol, z_idx, cfg):
    # Ejecuta el modelo CNN sobre un corte y retorna máscara binaria y mapa de probabilidad
    if modelo is None:
        h, w = vol['data_raw'].shape[:2]
        return np.zeros((h,w),dtype=bool), np.zeros((h,w)), np.zeros((h,w))
    device = next(modelo.parameters()).device; modelo.eval()
    plano = vol.get('plano', 'axial')
    data_norm = normalize_percentile(vol['data_raw'], cfg['pct_low'], cfg['pct_high'])
    sl_raw    = data_norm[z_idx,:,:] if plano=='sagittal' else data_norm[:,:,z_idx]
    sl_proc, _, _ = preprocess_slice_v3(sl_raw, cfg)
    th, tw    = cfg['target_size']
    img_r     = cv2.resize(sl_proc.astype(np.float32), (tw,th), interpolation=cv2.INTER_LINEAR)
    tensor    = torch.tensor((img_r-0.5)/0.5).unsqueeze(0).unsqueeze(0).float().to(device)
    with torch.no_grad(): logits = modelo(tensor)
    prob_map  = torch.sigmoid(logits).squeeze().cpu().numpy()
    h_orig, w_orig = sl_proc.shape
    prob_orig = cv2.resize(prob_map.astype(np.float32), (w_orig,h_orig), interpolation=cv2.INTER_LINEAR)
    return prob_orig>0.5, prob_orig, sl_proc


def comparar_clasico_vs_cnn(vol, z_idx, cfg, modelo):
    # Compara segmentación clásica v3 vs CNN — figura 2x3 con mapa TP/FP/FN
    mascara_v3, sl_proc = generar_pseudo_etiqueta(vol, z_idx, cfg)
    if modelo is not None:
        mascara_cnn, prob_map, _ = inferir_cnn(modelo, vol, z_idx, cfg)
    else:
        mascara_cnn = np.zeros_like(mascara_v3, dtype=bool); prob_map = np.zeros_like(mascara_v3, dtype=float)
    if mascara_cnn.shape != mascara_v3.shape:
        mascara_cnn = cv2.resize(mascara_cnn.astype(np.uint8),
                                  (mascara_v3.shape[1],mascara_v3.shape[0]),
                                  interpolation=cv2.INTER_NEAREST).astype(bool)
    a_v3 = mascara_v3.astype(bool); a_cnn = mascara_cnn.astype(bool)
    TP=(a_v3&a_cnn).sum(); FP=(~a_v3&a_cnn).sum(); FN=(a_v3&~a_cnn).sum(); TN=(~a_v3&~a_cnn).sum()
    dice_v=2*TP/(2*TP+FP+FN+1e-8); iou_v=TP/(TP+FP+FN+1e-8)
    sens=TP/(TP+FN+1e-8); espec=TN/(TN+FP+1e-8)
    print(f'\n  {vol["id"]} (z={z_idx})  Dice={dice_v:.3f}  IoU={iou_v:.3f}  Sens={sens:.3f}  Espec={espec:.3f}')

    diff_map = np.zeros((*a_v3.shape,3), dtype=float)
    diff_map[a_v3&a_cnn]   = [0,1,0]   # TP verde
    diff_map[~a_v3&a_cnn]  = [1,0,0]   # FP rojo
    diff_map[a_v3&~a_cnn]  = [0,0,1]   # FN azul

    sl_n = sl_proc/(sl_proc.max()+1e-8)
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes[0,0].imshow(sl_n.T, cmap='gray', origin='lower'); axes[0,0].set_title('Imagen preprocesada v3')
    axes[0,1].imshow(sl_n.T, cmap='gray', origin='lower', alpha=0.6)
    axes[0,1].imshow(np.ma.masked_where(~a_v3.T,a_v3.T), cmap='Greens', alpha=0.7, origin='lower')
    axes[0,1].set_title('Máscara v3 (pseudo-GT)')
    axes[0,2].imshow(sl_n.T, cmap='gray', origin='lower', alpha=0.6)
    axes[0,2].imshow(np.ma.masked_where(~a_cnn.T,a_cnn.T), cmap='Oranges', alpha=0.7, origin='lower')
    axes[0,2].set_title('Predicción CNN')
    axes[1,0].imshow(sl_n.T, cmap='gray', origin='lower', alpha=0.6)
    axes[1,0].imshow(np.ma.masked_where(~a_v3.T,a_v3.T.astype(float)*0.6), alpha=0.7, origin='lower', cmap='Greens')
    axes[1,0].set_title('Overlay v3')
    axes[1,1].imshow(sl_n.T, cmap='gray', origin='lower', alpha=0.6)
    axes[1,1].imshow(np.ma.masked_where(~a_cnn.T,a_cnn.T.astype(float)*0.6), alpha=0.7, origin='lower', cmap='Oranges')
    axes[1,1].set_title('Overlay CNN')
    axes[1,2].imshow(sl_n.T, cmap='gray', origin='lower', alpha=0.4)
    axes[1,2].imshow(diff_map.transpose(1,0,2), origin='lower', alpha=0.8)
    axes[1,2].legend(handles=[mpatches.Patch(color='green',label=f'TP={TP}'),
                                mpatches.Patch(color='red',  label=f'FP={FP}'),
                                mpatches.Patch(color='blue', label=f'FN={FN}')], fontsize=8)
    axes[1,2].set_title(f'Mapa TP/FP/FN\nDice={dice_v:.3f}  IoU={iou_v:.3f}')
    for ax in axes.flatten(): ax.axis('off')
    plt.suptitle(f'Clásico v3 vs CNN · {vol["id"]} (z={z_idx})', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTADOS_DIR, f'H_comparacion_cnn_{vol["id"]}.png'), dpi=100)
    plt.show()
    return {'dice':dice_v,'iou':iou_v,'sens':sens,'espec':espec}

print('OK  Funciones CNN de inferencia y comparación definidas.')

In [ ]:
print('Ejecutando comparación clásico v3 vs CNN para volúmenes representativos...')
comparaciones_cnn = {}
tipos_vistos = set()
for vol in volumenes:
    if vol['tipo'] not in tipos_vistos and vol['id'] in resultados_etapa3:
        tipos_vistos.add(vol['tipo'])
        z_best = resultados_etapa3[vol['id']]['z_best']
        try:
            res_cmp = comparar_clasico_vs_cnn(vol, z_best, CONFIG, modelo_cnn)
            comparaciones_cnn[vol['id']] = res_cmp
        except Exception as e:
            print(f'ERROR en {vol["id"]}: {e}')
print('OK  Comparaciones CNN completadas.')

### Visualización detallada de '05-badrecon' y '56-good'
A continuación, se muestra una comparación detallada de la segmentación clásica v3 contra la predicción de la CNN para '05-badrecon' y '56-good', utilizando sus 'mejores cortes' determinados por el pipeline.

In [ ]:
print('\n--- Comparando 05-badrecon ---')
vol_05 = next((v for v in volumenes if '05-badrecon' in v['id']), None)
if vol_05 and '05-badrecon' in resultados_etapa3:
    z_best_05 = resultados_etapa3['05-badrecon']['z_best']
    comparar_clasico_vs_cnn(vol_05, z_best_05, CONFIG, modelo_cnn)
else:
    print('AVISO: 05-badrecon no encontrado o no procesado en Etapa 3.')

print('\n--- Comparando 56-good ---')
vol_56 = next((v for v in volumenes if '56-good' in v['id']), None)
if vol_56 and '56-good' in resultados_etapa3:
    z_best_56 = resultados_etapa3['56-good']['z_best']
    comparar_clasico_vs_cnn(vol_56, z_best_56, CONFIG, modelo_cnn)
else:
    print('AVISO: 56-good no encontrado o no procesado en Etapa 3.')

---
## Sección I — Evaluación de Métricas

Tabla pandas completa para todos los volúmenes comparando clásico v3 vs CNN:
Dice, IoU, Sensibilidad, Especificidad, Accuracy, diámetros y clasificaciones.

In [ ]:
def evaluar_metricas_completo(volumenes, modelo, cfg, resultados_etapa3, resultados_etapa45):
    # Tabla de métricas completa para todos los volúmenes: Dice, IoU, Sens, Espec, Acc
    filas = []
    for vol in volumenes:
        vid = vol['id']
        if vid not in resultados_etapa3: continue
        r3   = resultados_etapa3[vid]; z_b = r3['z_best']; sp = r3['spacing']
        mascara_v3, sl_proc = generar_pseudo_etiqueta(vol, z_b, cfg)
        a_v3 = mascara_v3.astype(bool)
        d_clas = resultados_etapa45.get(vid, {}).get('diam_final', 0.0)
        c_clas = resultados_etapa45.get(vid, {}).get('clasificacion', 'N/A')

        if modelo is not None:
            try:
                mascara_cnn, _, _ = inferir_cnn(modelo, vol, z_b, cfg)
                if mascara_cnn.shape != a_v3.shape:
                    mascara_cnn = cv2.resize(mascara_cnn.astype(np.uint8),
                                              (a_v3.shape[1],a_v3.shape[0]),
                                              interpolation=cv2.INTER_NEAREST).astype(bool)
                a_cnn    = mascara_cnn.astype(bool)
                d_cnn, _ = min_bounding_rect_width(a_cnn, sp[0], sp[1])
                c_cnn, _ = diagnose(d_cnn, cfg)
            except Exception:
                a_cnn=np.zeros_like(a_v3); d_cnn=0.0; c_cnn='Error'
        else:
            a_cnn=np.zeros_like(a_v3); d_cnn=0.0; c_cnn='N/A'

        TP=(a_v3&a_cnn).sum(); FP=(~a_v3&a_cnn).sum()
        FN=(a_v3&~a_cnn).sum(); TN=(~a_v3&~a_cnn).sum()
        filas.append({'Volumen':vid,'Tipo':vol['tipo'],
                      'Dice':round(2*TP/(2*TP+FP+FN+1e-8),3),
                      'IoU':round(TP/(TP+FP+FN+1e-8),3),
                      'Sens':round(TP/(TP+FN+1e-8),3),'Espec':round(TN/(TN+FP+1e-8),3),
                      'Acc':round((TP+TN)/(TP+TN+FP+FN+1e-8),3),
                      'D_clasico_mm':round(d_clas,2),'D_CNN_mm':round(d_cnn,2),
                      'DeltaD_mm':round(abs(d_clas-d_cnn),2),
                      'Clasif_clas':c_clas,'Clasif_CNN':c_cnn})

    df_met = pd.DataFrame(filas)
    display(df_met)
    df_met.to_csv(os.path.join(RESULTADOS_DIR, 'I_metricas_completas.csv'), index=False)
    if len(df_met)==0: return df_met

    fig, axes = plt.subplots(2, 2, figsize=(14, 11))
    x = np.arange(len(df_met)); w=0.35
    tipo_colors = {'hospital_good':'limegreen','hospital_bad':'tomato','JC':'deepskyblue'}
    col_tipos = [tipo_colors.get(t,'gray') for t in df_met['Tipo']]

    axes[0,0].bar(x-w/2, df_met['Dice'], w, color='steelblue',  label='Dice')
    axes[0,0].bar(x+w/2, df_met['IoU'],  w, color='darkorange', label='IoU')
    axes[0,0].set_xticks(x); axes[0,0].set_xticklabels(df_met['Volumen'], rotation=45, ha='right', fontsize=7)
    axes[0,0].set_ylim(0,1.05); axes[0,0].set_ylabel('Métrica')
    axes[0,0].set_title('Dice e IoU por volumen'); axes[0,0].legend(); axes[0,0].grid(axis='y',alpha=0.3)

    d_max = max(df_met['D_clasico_mm'].max(), df_met['D_CNN_mm'].max())+2
    axes[0,1].scatter(df_met['D_clasico_mm'], df_met['D_CNN_mm'], c=col_tipos, s=80, zorder=3)
    axes[0,1].plot([0,d_max],[0,d_max],'w--',lw=1.5,label='Identidad')
    for _,row in df_met.iterrows(): axes[0,1].annotate(row['Volumen'],(row['D_clasico_mm'],row['D_CNN_mm']),fontsize=6)
    axes[0,1].set_xlabel('D clásico (mm)'); axes[0,1].set_ylabel('D CNN (mm)')
    axes[0,1].set_title('Correlación diámetros: clásico vs CNN'); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

    axes[1,0].bar(x-w/2, df_met['D_clasico_mm'], w, color=col_tipos, label='Clásico')
    axes[1,0].bar(x+w/2, df_met['D_CNN_mm'],     w, color=col_tipos, alpha=0.5, label='CNN')
    for th_val,th_col,th_lab in [(CONFIG['mild_mm'],'gold','Leve'),(CONFIG['moderate_mm'],'orange','Moderada'),(CONFIG['severe_mm'],'red','Severa')]:
        axes[1,0].axhline(th_val, color=th_col, lw=1.5, ls='--', label=f'{th_lab} {th_val}mm')
    axes[1,0].set_xticks(x); axes[1,0].set_xticklabels(df_met['Volumen'], rotation=45, ha='right', fontsize=7)
    axes[1,0].set_ylabel('Diámetro atrial (mm)')
    axes[1,0].set_title('Diámetros con umbrales clínicos'); axes[1,0].legend(fontsize=7); axes[1,0].grid(axis='y',alpha=0.3)

    cats = ['Normal','VM Leve','VM Moderada','VM Severa']
    cat_idx = {c:i for i,c in enumerate(cats)}
    conf_mat = np.zeros((4,4),dtype=int)
    for _,row in df_met.iterrows():
        i=cat_idx.get(row['Clasif_clas'],-1); j=cat_idx.get(row['Clasif_CNN'],-1)
        if i>=0 and j>=0: conf_mat[i,j]+=1
    axes[1,1].imshow(conf_mat, cmap='Blues', aspect='auto')
    axes[1,1].set_xticks(range(4)); axes[1,1].set_yticks(range(4))
    axes[1,1].set_xticklabels(cats, rotation=30, ha='right', fontsize=8)
    axes[1,1].set_yticklabels(cats, fontsize=8)
    axes[1,1].set_xlabel('Clasificación CNN'); axes[1,1].set_ylabel('Clasificación Clásico')
    axes[1,1].set_title('Matriz de confusión categórica')
    for i in range(4):
        for j in range(4): axes[1,1].text(j,i,str(conf_mat[i,j]),ha='center',va='center',fontsize=10)

    plt.suptitle('Sección I — Evaluación de Métricas · Clásico v3 vs CNN', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTADOS_DIR, 'I_metricas_graficas.png'), dpi=100)
    plt.show()
    return df_met

In [ ]:
print('Ejecutando evaluación de métricas completa...')
df_metricas_final = evaluar_metricas_completo(
    volumenes, modelo_cnn, CONFIG, resultados_etapa3, resultados_etapa45)
print('OK  Evaluación completada.')

---
## Sección J — Procesamiento Completo de los 10 Volúmenes

Mini-reporte de 8 paneles por volumen + tabla resumen comparativa.

In [ ]:
def procesar_todos_los_volumenes(volumenes, modelo, cfg):
    # Mini-reporte de 8 paneles por volumen (orientado + bilateral) + tabla resumen
    resumen_batch = []
    for vol in volumenes:
        print(f'\n{"="*58}\n  PROCESANDO: {vol["id"]}  ({vol["tipo"]})\n{"="*58}')
        try:
            z_best, sl_o, sp_2d, fix = get_best_slice_oriented(vol, cfg)
            sp0, sp1 = sp_2d
            sl_proc, snr_v, sig_v = preprocess_slice_v3(sl_o, cfg)
            t_b = threshold_otsu(sl_proc)
            bm  = remove_small_objects(sl_proc > t_b, min_size=100)

            lv, rv, mid = segment_bilateral(sl_proc, bm, cfg)
            _, _, _, _, _, nf, nb_val = detect_midline(sl_proc, bm)

            wl, rl = min_bounding_rect_width(lv, sp0, sp1)
            wr, rr = min_bounding_rect_width(rv, sp0, sp1)

            # Consistencia multiplanar (orientada)
            n_cross = cfg['cross_slices']; wmp = []
            n_ax = vol['data_raw'].shape[0] if vol.get('plano')=='sagittal' else vol['data_raw'].shape[2]
            for dz in range(-n_cross, n_cross + 1):
                z_c = z_best + dz
                if not (0 <= z_c < n_ax): continue
                try:
                    sl_c = extract_oriented_at(vol, z_c, cfg)
                    sp_c, _, _ = preprocess_slice_v3(sl_c, cfg)
                    bm_c = remove_small_objects(sp_c > threshold_otsu(sp_c), min_size=100)
                    lc, rc, _ = segment_bilateral(sp_c, bm_c, cfg)
                    wlc, _ = min_bounding_rect_width(lc, sp0, sp1)
                    wrc, _ = min_bounding_rect_width(rc, sp0, sp1)
                    w_c = max(wlc, wrc)
                    if w_c > 0: wmp.append((z_c, w_c))
                except Exception: pass

            ws_vals = [w for _, w in wmp]
            mean_w  = float(np.mean(ws_vals)) if ws_vals else max(wl, wr)
            cv_v    = float(np.std(ws_vals)/(mean_w+1e-8)) if len(ws_vals) > 1 else 0.0
            cons    = cv_v < cfg['cv_thresh']
            w_final = mean_w if cons else max(wl, wr)
            clasif, col_cl = diagnose(w_final, cfg)

            fig, axes = plt.subplots(2, 4, figsize=(20, 9))
            fig.patch.set_facecolor('#1a1a2e')
            for ax in axes.flatten(): ax.set_facecolor('#1a1a2e')
            def _show(ax, img, title, cmap='gray'):
                ax.imshow(img.T if img.ndim == 2 else img, cmap=cmap, origin='lower')
                ax.set_title(title, fontsize=9, color='white'); ax.axis('off')

            _show(axes[0,0], sl_o,    f'1. Original orientado idx={z_best}\nSNR={snr_v:.1f} fix={fix}')
            _show(axes[0,1], sl_proc, f'2. Post-proceso v3\nsigma={sig_v}')
            axes[0,2].imshow(sl_proc.T, cmap='gray', origin='lower')
            axes[0,2].axvline(mid, color='cyan', lw=2, ls='--')
            if nf is not None: axes[0,2].axvline(nf, color='lime', lw=1.5, ls=':')
            if nb_val is not None: axes[0,2].axvline(nb_val, color='yellow', lw=1.5, ls=':')
            axes[0,2].set_title(f'3. Cisura interhemisférica\ncisura x={mid}', fontsize=9, color='white')
            axes[0,2].axis('off')
            overlay_bilateral(axes[0,3], sl_proc, lv, rv)
            axes[0,3].set_title(f'4. Segmentacion\nL={int(lv.sum())}px  R={int(rv.sum())}px', fontsize=9, color='white')
            axes[0,3].axis('off')

            _show(axes[1,0], sl_proc, f'5. Ventrículo Izq\n{wl:.2f} mm')
            axes[1,0].imshow(np.ma.masked_where(~lv.T,lv.T), cmap='Blues', alpha=0.7, origin='lower')
            _show(axes[1,1], sl_proc, f'6. Ventrículo Der\n{wr:.2f} mm')
            axes[1,1].imshow(np.ma.masked_where(~rv.T,rv.T), cmap='Reds', alpha=0.7, origin='lower')

            _show(axes[1,2], sl_proc, f'7. Medición diámetro atrial\nL={wl:.1f}  R={wr:.1f} mm')
            axes[1,2].imshow(np.ma.masked_where(~lv.T,lv.T.astype(float)*0.7), cmap='Blues', alpha=0.5, origin='lower')
            axes[1,2].imshow(np.ma.masked_where(~rv.T,rv.T.astype(float)*0.7), cmap='Reds',  alpha=0.5, origin='lower')
            draw_rect_on_ax(axes[1,2], rl, 'deepskyblue', f'L={wl:.1f}mm')
            draw_rect_on_ax(axes[1,2], rr, 'tomato',      f'R={wr:.1f}mm')
            if rl is not None or rr is not None: axes[1,2].legend(fontsize=6, loc='lower left')

            if wmp:
                zs_mp, ws_mp = zip(*wmp)
                col_mp = ['limegreen' if w<10 else 'gold' if w<12.5 else 'orange' if w<15 else 'red' for w in ws_mp]
                axes[1,3].bar(range(len(zs_mp)), ws_mp, color=col_mp)
                for th_val,th_col in [(cfg['mild_mm'],'gold'),(cfg['moderate_mm'],'orange'),(cfg['severe_mm'],'red')]:
                    axes[1,3].axhline(th_val, color=th_col, lw=1, ls='--')
                axes[1,3].axhline(w_final, color='white', lw=2, label=f'Media={w_final:.1f}mm')
                axes[1,3].set_xticks(range(len(zs_mp)))
                axes[1,3].set_xticklabels([f'z{z}' for z in zs_mp], fontsize=6, color='white')
                axes[1,3].tick_params(colors='white'); axes[1,3].set_facecolor('#1a1a2e')
                axes[1,3].set_ylabel('mm', color='white')
                axes[1,3].set_title(f'8. Multiplanar +/-{n_cross}\nCV={cv_v*100:.1f}%', fontsize=9, color='white')
                axes[1,3].legend(fontsize=7)
            else:
                axes[1,3].text(0.5,0.5,'Sin datos\nmultiplanares',ha='center',va='center',
                               color='white',transform=axes[1,3].transAxes); axes[1,3].axis('off')

            plt.suptitle(f'{vol["id"]} ({vol["tipo"]}) — {clasif} ({w_final:.2f} mm)',
                         fontsize=14, fontweight='bold', color=col_cl)
            plt.tight_layout()
            plt.savefig(os.path.join(RESULTADOS_DIR, f'J_batch_{vol["id"]}.png'), dpi=100, facecolor='#1a1a2e')
            plt.show()

            resumen_batch.append({'Archivo':vol['id'],'Tipo':vol['tipo'],'z_best':z_best,
                'SNR':round(snr_v,2),'Cisura':mid,'D_izq_mm':round(wl,2),'D_der_mm':round(wr,2),
                'D_final':round(w_final,2),'CV_pct':round(cv_v*100,1),'Clasif':clasif})
        except Exception as e:
            print(f'ERROR procesando {vol["id"]}: {e}')
            import traceback; traceback.print_exc()

    df_batch = pd.DataFrame(resumen_batch)
    print('\n' + '='*65 + '\n  TABLA RESUMEN — TODOS LOS VOLÚMENES\n' + '='*65)
    display(df_batch)
    df_batch.to_csv(os.path.join(RESULTADOS_DIR, 'J_resumen_batch.csv'), index=False)

    # MOSAICO RESUMEN del batch (medicion con cajas)
    def _render_batch(ax, vol):
        z_b, sl_ob, sp2, fx = get_best_slice_oriented(vol, cfg)
        spx, _, _ = preprocess_slice_v3(sl_ob, cfg)
        bmx = remove_small_objects(spx > threshold_otsu(spx), min_size=100)
        lvx, rvx, _ = segment_bilateral(spx, bmx, cfg)
        wlx, rlx = min_bounding_rect_width(lvx, sp2[0], sp2[1])
        wrx, rrx = min_bounding_rect_width(rvx, sp2[0], sp2[1])
        overlay_bilateral(ax, spx, lvx, rvx, alpha=0.55)
        draw_rect_on_ax(ax, rlx, 'deepskyblue', None); draw_rect_on_ax(ax, rrx, 'tomato', None)
        cl, cc = diagnose(max(wlx, wrx), cfg)
        ax.set_title(f'{vol["id"]}\nD={max(wlx,wrx):.1f}mm {cl}', fontsize=8, color=cc)
    mosaico_resumen(volumenes, _render_batch,
        'RESUMEN BATCH — Medición diámetro atrial (L=azul · R=rojo · cajas=Cardoza)',
        'RESUMEN_batch_medicion.png')
    return df_batch


In [ ]:
print('Ejecutando procesamiento batch completo de todos los volúmenes...')
df_batch_final = procesar_todos_los_volumenes(volumenes, modelo_cnn, CONFIG)
print('OK  Batch completado.')

---
## Sección L — Análisis Prioritario: t2-t25 y 05-badrecon

Procesamiento especializado para los dos casos de presentación con parámetros
optimizados: t2-t25 (JC axial, alta calidad) y 05-badrecon (hospital, baja
calidad con NLM adicional).

---
## Celda de CALIBRACIÓN RÁPIDA (elige los parámetros que se vean como la imagen 2)

Esta celda **no cambia el pipeline**: solo prueba varias combinaciones de parámetros sobre
**un volumen** (por defecto el primero JC-axial que encuentre) y dibuja una **rejilla** con el
resultado de cada combinación. Observa cuál reproduce la **banda ventricular completa**
(asta frontal + cuerpo + atrio, azul/rojo) como en tu figura de referencia y **copia esos
valores al `CONFIG`** (las claves se imprimen debajo de cada panel).

- Si quieres calibrar otro volumen, cambia `VOL_CALIB` (subcadena del id, p.ej. `'t2-t24'`).
- Ejecuta **después** de cargar volúmenes y definir las funciones v3.

In [ ]:
# === CALIBRACIÓN RÁPIDA DE SEGMENTACIÓN ===
# Prueba combinaciones de (seed_pct, grow_floor_pct, max_grow_frac, close_r) y muestra
# una rejilla. NO modifica CONFIG: solo te ayuda a ELEGIR los valores que mejor lucen.

VOL_CALIB = 't2-t24'   # <-- cambia por el volumen que quieras calibrar (subcadena del id)

# Rejilla de parámetros a probar (edita libremente):
GRID = [
    # (seed_pct, inner_erode, atrium_post_frac, close_r)
    (91, 0.18, 0.62, 3),   # <- valores por defecto (recomendados)
    (93, 0.18, 0.62, 3),   # semilla mas estricta (ventriculo mas chico/limpio)
    (89, 0.16, 0.66, 3),   # semilla mas amplia (llena mas)
    (91, 0.22, 0.62, 4),   # mas erosion (excluye mas LCR periferico)
    (91, 0.14, 0.58, 3),   # menos erosion (mas region interna)
    (88, 0.18, 0.62, 4),   # mas permisivo
]

_vol = next((v for v in volumenes if VOL_CALIB in v['id']), None)
if _vol is None:
    print(f'AVISO: no encontré "{VOL_CALIB}" en volumenes. Volúmenes disponibles:')
    for v in volumenes: print('   -', v['id'])
else:
    # corte + preprocesado UNA sola vez (suave, estilo t25-bueno)
    cfg_base = {**CONFIG,
        'clahe_clip': 0.002, 'bilateral_d': 15, 'bilateral_sc': 100, 'bilateral_ss': 100}
    z_best, sl_raw, sp_2d = get_best_slice(_vol, cfg_base)
    fix = _resolver_fix(_vol, sl_raw, cfg_base)
    sl_o = apply_orientation(sl_raw, fix)
    sl_proc, snr_v, sig_v = preprocess_slice_v3(sl_o, cfg_base)
    t_b = threshold_otsu(sl_proc)
    bm  = remove_small_objects(binary_fill_holes(sl_proc > t_b), min_size=100)
    mid, *_ = detect_midline(sl_proc, bm)

    n = len(GRID); cols = 3; rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols*4.2, rows*4.2))
    fig.patch.set_facecolor('#0d1b2a'); axes = np.array(axes).reshape(-1)
    print(f'Calibrando {_vol["id"]}  (z={z_best}, midline={mid}, SNR={snr_v:.1f})\n')
    for k, (sp, ie, apf, cr) in enumerate(GRID):
        cfg_try = {**cfg_base, 'seed_pct': sp, 'inner_erode': ie,
                   'atrium_post_frac': apf, 'close_r': cr}
        lv, rv = segment_ventricles(sl_proc, bm, mid, cfg_try)
        ax = axes[k]; ax.set_facecolor('#0d1b2a')
        overlay_bilateral(ax, sl_proc, lv, rv, alpha=0.85)
        ax.axvline(mid, color='cyan', lw=1, ls='--', alpha=0.5)
        ax.set_title(f"seed_pct={sp}  inner_erode={ie}\natrium={apf}  close_r={cr}\n"
                     f"L={int(lv.sum())}px  R={int(rv.sum())}px",
                     fontsize=8, color='white')
        ax.axis('off')
    for j in range(n, len(axes)): axes[j].axis('off')
    plt.suptitle(f'CALIBRACIÓN — {_vol["id"]} — elige el panel que luce como la imagen 2',
                 color='white', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTADOS_DIR, f'CALIBRACION_{VOL_CALIB}.png'),
                dpi=140, facecolor='#0d1b2a')
    plt.show()
    print('\nCuando identifiques el mejor panel, copia esos 4 valores al CONFIG:')
    print("   CONFIG['seed_pct'], CONFIG['inner_erode'], CONFIG['atrium_post_frac'], CONFIG['close_r']")


In [ ]:
# === ANÁLISIS PRIORITARIO: t2-t25 (JC axial) ===
# Parámetros mejorados para máxima cobertura ventricular
CFG_T25 = {**CONFIG,
    'clahe_clip': 0.002,
    'bilateral_d': 15, 'bilateral_sc': 100, 'bilateral_ss': 100,
    'ventricle_pct': 82,
    'close_r': 4,
    'min_size': 50,
}
vol_t25 = next((v for v in volumenes if 't2-t25' in v['id']), None)
if vol_t25 is None:
    print('AVISO: t2-t25 no encontrado en volumenes')
else:
    print(f'=== ANÁLISIS PRIORITARIO: {vol_t25["id"]} ===')
    z_best, sl_raw, sp_2d = get_best_slice(vol_t25, CFG_T25)
    sp0, sp1 = sp_2d
    sl_proc, snr_v, sig_v = preprocess_slice_v3(sl_raw, CFG_T25)
    t_b = threshold_otsu(sl_proc)
    bm  = remove_small_objects(sl_proc > t_b, min_size=100)
    mid, _, col_sm, _, _, nf, nb_val = detect_midline(sl_proc, bm)
    lv, rv = segment_ventricles(sl_proc, bm, mid, CFG_T25)
    wl, rl = min_bounding_rect_width(lv, sp0, sp1)
    wr, rr = min_bounding_rect_width(rv, sp0, sp1)
    w_fin  = max(wl, wr)
    clasif, col_cl = diagnose(w_fin, CFG_T25)

    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.patch.set_facecolor('#0d1b2a')
    for ax in axes.flatten(): ax.set_facecolor('#0d1b2a')

    def _sp(ax, img, title, cmap='gray'):
        ax.imshow(img.T, cmap=cmap, origin='lower')
        ax.set_title(title, fontsize=10, color='white', pad=6); ax.axis('off')

    _sp(axes[0,0], sl_raw,  f'1. Original — corte axial idx={z_best}\nSNR={snr_v:.1f}')
    _sp(axes[0,1], sl_proc, '2. Preprocesado mejorado\nCLAHE clip=0.002  Bilateral d=15  sc=100')

    axes[0,2].imshow(sl_proc.T, cmap='gray', origin='lower')
    axes[0,2].axvline(mid, color='cyan', lw=2, ls='--', label=f'Cisura x={mid}')
    if nf: axes[0,2].axvline(nf, color='lime', lw=1.5, ls=':', label='Frontal')
    if nb_val: axes[0,2].axvline(nb_val, color='yellow', lw=1.5, ls=':', label='Occipital')
    axes[0,2].legend(fontsize=7, loc='upper right', facecolor='#0d1b2a', labelcolor='white')
    axes[0,2].set_title(f'3. Deteccion de cisura\nmidline={mid}', fontsize=10, color='white')
    axes[0,2].axis('off')

    overlay_bilateral(axes[1,0], sl_proc, lv, rv, alpha=0.85)
    axes[1,0].set_title(f'4. Segmentacion completa\nL={lv.sum()}px  R={rv.sum()}px', fontsize=10, color='white')
    axes[1,0].axis('off')

    axes[1,1].imshow(sl_proc.T, cmap='gray', origin='lower')
    axes[1,1].imshow(np.ma.masked_where(~lv.T, lv.T.astype(float)*0.7), cmap='Blues', alpha=0.6, origin='lower')
    axes[1,1].imshow(np.ma.masked_where(~rv.T, rv.T.astype(float)*0.7), cmap='Reds',  alpha=0.6, origin='lower')
    draw_rect_on_ax(axes[1,1], rl, 'deepskyblue', f'L={wl:.1f}mm')
    draw_rect_on_ax(axes[1,1], rr, 'tomato',      f'R={wr:.1f}mm')
    axes[1,1].legend(fontsize=8, loc='lower left', facecolor='#0d1b2a', labelcolor='white')
    axes[1,1].set_title(f'5. Medicion diametro atrial\nL={wl:.2f}mm  R={wr:.2f}mm', fontsize=10, color='white')
    axes[1,1].axis('off')

    diag_txt = (f'DIAGNOSTICO: {clasif}\n\n'
                f'Diametro maximo: {w_fin:.2f} mm\n'
                f'Ventriculo izq:  {wl:.2f} mm\n'
                f'Ventriculo der:  {wr:.2f} mm\n'
                f'SNR estimado:    {snr_v:.1f}\n\n'
                f'Criterio Cardoza 1988:\n'
                f'  Normal    < 10 mm\n'
                f'  Leve      10-12 mm\n'
                f'  Moderada  12-15 mm\n'
                f'  Severa    > 15 mm')
    axes[1,2].text(0.05, 0.95, diag_txt, transform=axes[1,2].transAxes,
                   fontsize=11, va='top', color='white', fontfamily='monospace',
                   bbox=dict(boxstyle='round', facecolor=col_cl, alpha=0.3))
    axes[1,2].set_title('6. Resultado diagnostico', fontsize=10, color='white'); axes[1,2].axis('off')

    plt.suptitle(f'ANALISIS PRIORITARIO — t2-t25 (JC axial) — {clasif} ({w_fin:.2f} mm)',
                 fontsize=13, fontweight='bold', color=col_cl)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTADOS_DIR, 'PRIORIDAD_t2t25.png'), dpi=150, facecolor='#0d1b2a')
    plt.show()
    print(f'OK  t2-t25: {clasif} — L={wl:.2f}mm  R={wr:.2f}mm  D={w_fin:.2f}mm')

In [ ]:
# === ANÁLISIS PRIORITARIO: 05-badrecon (hospital_bad, sagittal) ===
# Pipeline con Non-Local Means para maxima claridad ventricular
CFG_05 = {**CONFIG,
    'clahe_clip': 0.003, 'clahe_nbins': 512,
    'bilateral_d': 15, 'bilateral_sc': 150, 'bilateral_ss': 120,
    'ventricle_pct': 80,
    'close_r': 5,
    'min_size': 60,
}
vol_05 = next((v for v in volumenes if '05-badrecon' in v['id']), None)
if vol_05 is None:
    print('AVISO: 05-badrecon no encontrado en volumenes')
else:
    print(f'=== ANÁLISIS PRIORITARIO: {vol_05["id"]} ===')
    z_best, sl_raw, sp_2d = get_best_slice(vol_05, CFG_05)
    sp0, sp1 = sp_2d

    sl_std, snr_std, _ = preprocess_slice_v3(sl_raw, CFG_05)

    # Non-Local Means denoising para maximo detalle ventricular
    sl_u8  = (np.clip(sl_raw, 0, 1) * 255).astype(np.uint8)
    sl_nlm = cv2.fastNlMeansDenoising(sl_u8, None, h=12, templateWindowSize=7, searchWindowSize=21)
    sl_nlm_f = sl_nlm.astype(np.float32) / 255.0
    sl_nlm_proc, snr_v, sig_v = preprocess_slice_v3(sl_nlm_f, CFG_05)

    t_b = threshold_otsu(sl_nlm_proc)
    bm  = remove_small_objects(sl_nlm_proc > t_b, min_size=100)
    lv  = segment_sagittal_ventricle(sl_nlm_proc, bm, CFG_05)
    rv  = np.zeros_like(lv)
    wl, rl = min_bounding_rect_width(lv, sp0, sp1)
    w_fin  = wl
    clasif, col_cl = diagnose(w_fin, CFG_05)

    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.patch.set_facecolor('#0d1b2a')
    for ax in axes.flatten(): ax.set_facecolor('#0d1b2a')

    def _sp2(ax, img, title, cmap='gray'):
        ax.imshow(img.T, cmap=cmap, origin='lower')
        ax.set_title(title, fontsize=10, color='white', pad=6); ax.axis('off')

    _sp2(axes[0,0], sl_raw,      f'1. Original reconstruccion\nSNR={snr_std:.1f}')
    _sp2(axes[0,1], sl_nlm_f,    '2. Despues de NLM\n(h=12  template=7  search=21)')
    _sp2(axes[0,2], sl_nlm_proc, '3. NLM + CLAHE + Bilateral\nnbins=512  d=15  sc=150')

    _sp2(axes[1,0], sl_std, '4. Clasico v3 (referencia)')

    overlay_solido_unico(axes[1,1], sl_nlm_proc, lv, color=(0.32,0.55,0.78), alpha=0.85)
    if rl is not None: draw_rect_on_ax(axes[1,1], rl, 'deepskyblue', f'D={wl:.1f}mm')
    axes[1,1].set_title(f'5. Segmentacion con NLM\nD={wl:.2f} mm', fontsize=10, color='white')
    axes[1,1].axis('off')

    diag_txt2 = (f'DIAGNOSTICO: {clasif}\n\n'
                 f'Diametro: {w_fin:.2f} mm\n'
                 f'SNR orig: {snr_std:.1f}\n\n'
                 f'Pipeline NLM:\n'
                 f'  NLM h=12\n'
                 f'  CLAHE nbins=512\n'
                 f'  Bilateral d=15\n\n'
                 f'Nota: baja calidad\nde reconstruccion.\nCaucion diagnostica.')
    axes[1,2].text(0.05, 0.95, diag_txt2, transform=axes[1,2].transAxes,
                   fontsize=11, va='top', color='white', fontfamily='monospace',
                   bbox=dict(boxstyle='round', facecolor=col_cl, alpha=0.3))
    axes[1,2].set_title('6. Resultado diagnostico', fontsize=10, color='white'); axes[1,2].axis('off')

    plt.suptitle(f'ANALISIS PRIORITARIO — 05-badrecon (hospital sagittal) — {clasif} ({w_fin:.2f} mm)',
                 fontsize=13, fontweight='bold', color=col_cl)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTADOS_DIR, 'PRIORIDAD_05badrecon.png'), dpi=150, facecolor='#0d1b2a')
    plt.show()
    print(f'OK  05-badrecon: {clasif} — D={w_fin:.2f}mm (NLM-enhanced)')

In [ ]:
# === VEREDICTO CNN — EVALUACIÓN INTEGRAL ===
sep70 = '=' * 70
print(sep70)
print('  VEREDICTO CNN — EVALUACION INTEGRAL')
print(sep70)
print()
print('ADVERTENCIA METODOLOGICA:')
print('  La CNN fue entrenada con pseudo-etiquetas del algoritmo clasico v3,')
print('  NO con anotaciones manuales de radiologos. El Dice mide concordancia')
print('  CNN-vs-clasico, no CNN-vs-verdad-radiologica.')
print()

if 'df_metricas_final' not in dir() or df_metricas_final.empty:
    print('  (Ejecutar Seccion I — evaluar_metricas_completo — primero)')
else:
    tabla_v = []
    for _, r in df_metricas_final.iterrows():
        estado = 'OK' if r['Dice'] > 0.60 else ('PARCIAL' if r['Dice'] > 0.30 else 'FALLO')
        concord = 'SI' if r['Clasif_clas'] == r['Clasif_CNN'] else 'NO'
        tabla_v.append({'Volumen': r['Volumen'], 'Tipo': r['Tipo'],
                        'Dice': r['Dice'], 'IoU': r['IoU'],
                        'D_clasico_mm': r['D_clasico_mm'], 'D_CNN_mm': r['D_CNN_mm'],
                        'DeltaD_mm': r['DeltaD_mm'],
                        'Clasif_clas': r['Clasif_clas'], 'Clasif_CNN': r['Clasif_CNN'],
                        'Concord_dx': concord, 'Estado': estado})
    df_v = pd.DataFrame(tabla_v)
    display(df_v)

    dice_mean  = df_v['Dice'].mean()
    iou_mean   = df_v['IoU'].mean()
    delta_mean = df_v['DeltaD_mm'].mean()
    conc_dx    = (df_v['Concord_dx'] == 'SI').sum()
    total_v    = len(df_v)
    ok_v       = (df_v['Estado'] == 'OK').sum()
    fallo_v    = (df_v['Estado'] == 'FALLO').sum()
    best_vols  = df_v.nlargest(3, 'Dice')['Volumen'].tolist()
    worst_vols = df_v.nsmallest(3, 'Dice')['Volumen'].tolist()
    best_tipos = df_v.groupby('Tipo')['Dice'].mean().sort_values(ascending=False)

    print('-' * 70)
    print('ESTADISTICAS GLOBALES:')
    print('-' * 70)
    print(f'  Dice promedio (CNN vs pseudo-etiquetas): {dice_mean:.3f}')
    print(f'  IoU promedio:                            {iou_mean:.3f}')
    print(f'  Delta diametro promedio |clasico-CNN|:   {delta_mean:.2f} mm')
    print(f'  Concordancia diagnostica: {conc_dx}/{total_v} ({100*conc_dx/total_v:.0f}%)')
    print(f'  Volumenes OK (Dice>0.60):    {ok_v}/{total_v}')
    print(f'  Volumenes FALLO (Dice<0.30): {fallo_v}/{total_v}')
    print()
    print(f'  MEJORES imagenes (Dice): {", ".join(best_vols)}')
    print(f'  PEORES  imagenes (Dice): {", ".join(worst_vols)}')
    print()
    print('  Dice promedio por tipo:')
    for tipo_k, dice_k in best_tipos.items():
        print(f'    {tipo_k:<20}: {dice_k:.3f}')

    # Graficos
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.patch.set_facecolor('#1a1a2e')
    for ax in axes: ax.set_facecolor('#1a1a2e')
    tipo_colors = {'hospital_good': 'limegreen', 'hospital_bad': 'tomato', 'JC': 'deepskyblue'}
    cols = [tipo_colors.get(t, 'gray') for t in df_v['Tipo']]

    axes[0].bar(range(total_v), df_v['Dice'], color=cols)
    axes[0].axhline(0.6, color='lime',   lw=1, ls='--', label='OK (0.60)')
    axes[0].axhline(0.3, color='orange', lw=1, ls='--', label='FALLO (0.30)')
    axes[0].set_xticks(range(total_v))
    axes[0].set_xticklabels(df_v['Volumen'], rotation=45, ha='right', fontsize=7, color='white')
    axes[0].set_ylim(0, 1.05); axes[0].set_ylabel('Dice', color='white')
    axes[0].tick_params(colors='white'); axes[0].legend(fontsize=7)
    axes[0].set_title('Dice CNN vs Pseudo-etiquetas', color='white', fontsize=10)

    x = np.arange(total_v); w_b = 0.35
    axes[1].bar(x - w_b/2, df_v['D_clasico_mm'], w_b, color='steelblue',  label='Clasico v3', alpha=0.9)
    axes[1].bar(x + w_b/2, df_v['D_CNN_mm'],     w_b, color='darkorange', label='CNN', alpha=0.9)
    for th_v, th_c in [(10, 'gold'), (12, 'orange'), (15, 'red')]:
        axes[1].axhline(th_v, color=th_c, lw=1, ls='--')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(df_v['Volumen'], rotation=45, ha='right', fontsize=7, color='white')
    axes[1].set_ylabel('mm', color='white'); axes[1].legend(fontsize=8)
    axes[1].tick_params(colors='white')
    axes[1].set_title('Diametro: Clasico vs CNN', color='white', fontsize=10)

    tipo_lista = list(best_tipos.index); dice_lista = list(best_tipos.values)
    axes[2].barh(tipo_lista, dice_lista, color=[tipo_colors.get(t, 'gray') for t in tipo_lista])
    axes[2].axvline(0.6, color='lime', lw=1.5, ls='--')
    axes[2].set_xlabel('Dice promedio', color='white')
    axes[2].tick_params(colors='white')
    axes[2].set_title('Rendimiento por tipo', color='white', fontsize=10)
    for i, v in enumerate(dice_lista):
        axes[2].text(v + 0.01, i, f'{v:.3f}', va='center', color='white', fontsize=9)

    handles = [mpatches.Patch(color=c, label=t) for t, c in tipo_colors.items()]
    fig.legend(handles=handles, loc='upper center', ncol=3, fontsize=9,
               facecolor='#1a1a2e', labelcolor='white')
    plt.suptitle('VEREDICTO CNN — Evaluacion Integral del Rendimiento', fontsize=12, color='white')
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.savefig(os.path.join(RESULTADOS_DIR, 'K_cnn_veredicto.png'), dpi=120, facecolor='#1a1a2e')
    plt.show()

    # Conclusion en lenguaje natural
    if dice_mean >= 0.70:
        cnn_status = 'CONVERGIO SATISFACTORIAMENTE'
        cnn_rec = 'RECOMENDADA como segundo verificador automatico'
    elif dice_mean >= 0.50:
        cnn_status = 'CONVERGIO PARCIALMENTE'
        cnn_rec = 'RECOMENDADA CON RESERVAS — util como guia visual'
    else:
        cnn_status = 'NO CONVERGIO de forma fiable'
        cnn_rec = 'NO RECOMENDADA sin re-entrenamiento con mas datos'

    print()
    print('=' * 70)
    print('  CONCLUSION EN LENGUAJE NATURAL')
    print('=' * 70)
    print(f'ESTADO DE LA CNN: {cnn_status}')
    print()
    print(f'La red Attention U-Net + EfficientNet-B0 fue entrenada con')
    print(f'pseudo-etiquetas del pipeline clasico v3 sobre {total_v} volumenes.')
    print()
    print(f'RENDIMIENTO OBSERVADO:')
    print(f'  Dice promedio: {dice_mean:.3f}  |  IoU: {iou_mean:.3f}')
    print(f'  Error diametro vs clasico: {delta_mean:.2f} mm promedio')
    print(f'  Concordancia diagnostica: {conc_dx}/{total_v}')
    print()
    print(f'MEJOR TIPO: {best_tipos.index[0]} (Dice {best_tipos.iloc[0]:.3f})')
    print(f'  Las imagenes de mejor SNR facilitan pseudo-etiquetas mas')
    print(f'  precisas y por ende mejor aprendizaje de la CNN.')
    print()
    print(f'PEOR TIPO: {best_tipos.index[-1]} (Dice {best_tipos.iloc[-1]:.3f})')
    print(f'  Los artefactos de reconstruccion degradan las pseudo-etiquetas')
    print(f'  y dificultan el aprendizaje. La CNN no distingue LCR real de')
    print(f'  brillos artificiosos por ruido.')
    print()
    print(f'LIMITACIONES:')
    print(f'  1. Sin ground truth radiologico — Dice vs pseudo-etiquetas.')
    print(f'  2. Dataset pequeno ({total_v} volumenes) insuficiente para fine-tuning robusto.')
    print(f'  3. Diferente perfil de intensidad entre tipos (sagital vs axial)')
    print(f'     puede afectar la generalizacion de la CNN entre grupos.')
    print(f'  4. Imbalance: JC tiene mejor calidad, hospital tiene peor.')
    print()
    print(f'RECOMENDACION: {cnn_rec}')
    print()
    print(f'Para uso clinico riguroso se recomienda:')
    print(f'  - Anotacion manual de 10-20 imagenes por radiologo especialista')
    print(f'  - Re-entrenamiento con augmentaciones (flips, rotaciones, gamma)')
    print(f'  - Validacion externa cruzada')
    print()
    print(f'VEREDICTO FINAL:')
    print(f'  El pipeline clasico v3 es el metodo mas confiable en este dataset.')
    print(f'  La CNN aporta valor como visualizador de segundo criterio y podria')
    print(f'  mejorar significativamente con anotaciones manuales adicionales.')
    print()
    print(sep70)

---
## Sección K — Conclusiones y Limitaciones

Reporte final formateado con resultados, conclusiones metodológicas y limitaciones.

In [ ]:
# SECCIÓN K — Reporte Final Formateado
sep = '=' * 68
print(sep)
print('         REPORTE FINAL — DIAGNÓSTICO DE VENTRICULOMEGALIA')
print('         Tecnológico de Monterrey · Procesamiento de Imágenes')
print(sep)
print('\nMETODOLOGÍA:')
print('  Pipeline clásico: Normalización percentil -> ROI automático ->')
print('    Difusión anisótropa -> CLAHE -> Bilateral -> Orientación ->')
print('    Cisura hemisférica -> Segmentación percentil 90 ->')
print('    Rectángulo mínimo -> Validación multiplanar +/- 3 cortes')
print()
print('  CNN: Attention U-Net (Oktay 2018) + EfficientNet-B0 (Tan 2019)')
print('    Transfer learning desde ImageNet | pseudo-etiquetas v3')
print()
print('-' * 68)
print('RESULTADOS POR VOLUMEN:')
print('-' * 68)
if 'df_batch_final' in dir() and not df_batch_final.empty:
    hdr = f"  {'Archivo':<20} {'Tipo':<14} {'D_final':>8} {'CV%':>6}  {'Clasif'}"
    print(hdr)
    for _, r in df_batch_final.iterrows():
        print(f"  {r['Archivo']:<20} {r['Tipo']:<14} {r['D_final']:>7.2f}mm {r['CV_pct']:>5.1f}%  {r['Clasif']}")
else:
    print('  (Ejecutar Sección J primero)')

print()
print('-' * 68)
print('ACCURACY DIAGNÓSTICA (clásico vs CNN):')
print('-' * 68)
if 'df_metricas_final' in dir() and not df_metricas_final.empty:
    conc = (df_metricas_final['Clasif_clas']==df_metricas_final['Clasif_CNN']).sum()
    total_m = len(df_metricas_final)
    print(f'  Concordancia clásico vs CNN: {conc}/{total_m} diagnósticos iguales')
    print(f'  Dice promedio (v3 vs CNN):   {df_metricas_final["Dice"].mean():.3f}')
else:
    print('  (Ejecutar Sección I primero)')

print()
print('-' * 68)
print('CONCLUSIONES:')
print('-' * 68)
print('  1. El pipeline clásico v3 produce segmentaciones anatómicamente')
print('     consistentes usando CLAHE + bilateral + separación hemisférica.')
print('  2. La CNN aprende a replicar las pseudo-etiquetas con transfer')
print('     learning desde ImageNet, demostrado en Raghu et al. 2019.')
print('  3. Las imágenes JC-axiales (mejor calidad) producen segmentaciones')
print('     más confiables (CV < 0.20) que las hospitalarias.')

print()
print('-' * 68)
print('LIMITACIONES:')
print('-' * 68)
print('  1. Sin ground truth: Dice calculado contra pseudo-etiquetas v3.')
print('  2. 10 imágenes insuficientes para fine-tuning robusto de la CNN.')
print('  3. Imágenes de mala reconstrucción: SNR bajo eleva el CV multiplanar.')
print('  4. La CNN no distingue artefactos de movimiento de verdadero LCR.')
print('  5. Orientación automática puede fallar en cortes extremos del volumen.')

print()
print('-' * 68)
print('PLAN DE TRABAJO FUTURO:')
print('-' * 68)
print('  - Anotación manual de al menos 3 imágenes para validación externa')
print('  - Aplicar nnU-Net (Isensee 2021) con pesos del desafío FeTA 2021')
print('  - Incorporar corrección de bias field (N4ITK, Tustison 2010)')
print('  - Segmentación 3D volumétrica (vs. 2D por corte actual)')
print('  - Dataset: solicitar datos del hospital con anotaciones de experto')
print()
print(sep)
print(f'  Resultados guardados en: {RESULTADOS_DIR}')
print(sep)

---
## Sección M — RESÚMENES PARA PRESENTACIÓN (alineados al objetivo SMART)
Genera una imagen-resumen por etapa, lista para diapositivas:
**Etapa 1** Base de datos · **Etapa 2** Reducción de ruido · **Etapa 3** Segmentación ·
**Etapas 4-5** Cuantificación y clasificación · **Validación** · **Trabajo a futuro** · **Conclusión**.
Todas las imágenes se guardan en la carpeta `resultados/`.


In [ ]:
# === SECCIÓN M — Cálculo unificado para todos los resúmenes de presentación ===
print('Generando datos unificados (orientados + bilaterales) para la presentación...')
pres_data = []
for vol in volumenes:
    try:
        z_b, sl_o, sp2, fx = get_best_slice_oriented(vol, CONFIG)
        sp0, sp1 = sp2
        sl_proc, snr, sig = preprocess_slice_v3(sl_o, CONFIG)
        bm = remove_small_objects(sl_proc > threshold_otsu(sl_proc), min_size=100)
        lv, rv, mid = segment_bilateral(sl_proc, bm, CONFIG)
        wl, rl = min_bounding_rect_width(lv, sp0, sp1)
        wr, rr = min_bounding_rect_width(rv, sp0, sp1)
        # diametro final: usa consistencia multiplanar de Etapa III si existe
        r3 = resultados_etapa3.get(vol['id'], {})
        w_final = r3.get('w_final') or max(wl, wr)
        cv = r3.get('cv', 0.0)
        clasif, color = diagnose(w_final, CONFIG)
        pres_data.append({'id':vol['id'],'tipo':vol['tipo'],'sl_o':sl_o,'sl_proc':sl_proc,
            'lv':lv,'rv':rv,'mid':mid,'rl':rl,'rr':rr,'wl':wl,'wr':wr,
            'w_final':w_final,'cv':cv,'clasif':clasif,'color':color,'snr':snr})
    except Exception as e:
        print(f'  AVISO {vol["id"]}: {e}')
print(f'OK  {len(pres_data)} volúmenes listos para resúmenes.')


In [ ]:
# === RESUMEN ETAPAS 4-5 — Cuantificación y Clasificación (Objetivos R y T) ===
n = len(pres_data); cols = min(5, max(1,n)); rows = math.ceil(n/cols)
fig = plt.figure(figsize=(cols*3.2, rows*3.2 + 2.2)); fig.patch.set_facecolor('#0d1b2a')
gs = fig.add_gridspec(rows+1, cols, height_ratios=[1]*rows + [0.7])
for i, d in enumerate(pres_data):
    ax = fig.add_subplot(gs[i//cols, i%cols]); ax.set_facecolor('#0d1b2a')
    overlay_bilateral(ax, d['sl_proc'], d['lv'], d['rv'], alpha=0.55)
    draw_rect_on_ax(ax, d['rl'], 'deepskyblue', None)
    draw_rect_on_ax(ax, d['rr'], 'tomato', None)
    ax.set_title(f'{d["id"]}\nD={d["w_final"]:.1f}mm · {d["clasif"]}', fontsize=8, color=d['color'])
    ax.axis('off')
# barra inferior: escala clinica con cada volumen marcado
axb = fig.add_subplot(gs[rows, :]); axb.set_facecolor('#0d1b2a')
for thr,col,etq,nx in [(0,'limegreen','Normal',CONFIG['mild_mm']),
                       (CONFIG['mild_mm'],'gold','Leve',CONFIG['moderate_mm']),
                       (CONFIG['moderate_mm'],'orange','Moderada',CONFIG['severe_mm']),
                       (CONFIG['severe_mm'],'red','Severa',25)]:
    axb.barh(0, nx-thr, left=thr, height=0.5, color=col, alpha=0.65, label=etq)
for d in pres_data:
    axb.axvline(d['w_final'], color='white', lw=1.2)
    axb.text(d['w_final'], 0.45, d['id'].replace('-badrecon','').replace('-good','').replace('t2-',''),
             rotation=90, fontsize=6, color='white', ha='center', va='bottom')
axb.set_xlim(0,25); axb.set_ylim(-0.4,1.2); axb.set_yticks([])
axb.set_xlabel('Diámetro atrial (mm) — Criterio Cardoza 1988', color='white')
axb.tick_params(colors='white'); axb.legend(loc='upper right', fontsize=7, ncol=4)
plt.suptitle('RESUMEN ETAPAS 4-5 — Cuantificación geométrica + Clasificación diagnóstica\n'
             '(Obj. R: cuantificar propiedades ventriculares · Obj. T: umbralización LCR en T2)',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTADOS_DIR, 'RESUMEN_etapa45_cuantificacion.png'), dpi=140, facecolor='#0d1b2a')
plt.show()

# tabla resumen
df_pres = pd.DataFrame([{'Volumen':d['id'],'Tipo':d['tipo'],'D_Izq_mm':round(d['wl'],2),
    'D_Der_mm':round(d['wr'],2),'D_Final_mm':round(d['w_final'],2),
    'CV_pct':round(d['cv']*100,1),'Clasificacion':d['clasif']} for d in pres_data])
display(df_pres)
df_pres.to_csv(os.path.join(RESULTADOS_DIR, 'RESUMEN_cuantificacion.csv'), index=False)


In [ ]:
# === RESUMEN VALIDACIÓN — Consistencia multiplanar (Obj. A y R) ===
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5)); fig.patch.set_facecolor('#0d1b2a')
ids = [d['id'] for d in pres_data]; cvs = [d['cv']*100 for d in pres_data]
cols_cv = ['limegreen' if c < CONFIG['cv_thresh']*100 else 'orange' for c in cvs]
ax1.bar(range(len(ids)), cvs, color=cols_cv); ax1.set_facecolor('#0d1b2a')
ax1.axhline(CONFIG['cv_thresh']*100, color='red', ls='--', lw=1.5, label=f'Umbral {CONFIG["cv_thresh"]*100:.0f}%')
ax1.set_xticks(range(len(ids))); ax1.set_xticklabels(ids, rotation=90, fontsize=7, color='white')
ax1.set_ylabel('CV inter-corte (%)', color='white'); ax1.tick_params(colors='white')
ax1.set_title('Variabilidad multiplanar por volumen', color='white'); ax1.legend(fontsize=8)
# distribucion de clasificaciones
from collections import Counter
cnt = Counter(d['clasif'] for d in pres_data)
cmap_cl = {'Normal':'limegreen','VM Leve':'gold','VM Moderada':'orange','VM Severa':'red'}
labels = list(cnt.keys()); vals = [cnt[k] for k in labels]
ax2.bar(labels, vals, color=[cmap_cl.get(k,'gray') for k in labels]); ax2.set_facecolor('#0d1b2a')
ax2.set_ylabel('Nº de volúmenes', color='white'); ax2.tick_params(colors='white')
ax2.set_title('Distribución de clasificación diagnóstica', color='white')
n_cons = sum(1 for d in pres_data if d['cv'] < CONFIG['cv_thresh'])
plt.suptitle(f'RESUMEN VALIDACIÓN — {n_cons}/{len(pres_data)} volúmenes consistentes (CV<{CONFIG["cv_thresh"]*100:.0f}%)\n'
             '(Obj. A: localización robusta · Obj. R: cuantificación reproducible)',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTADOS_DIR, 'RESUMEN_validacion.png'), dpi=140, facecolor='#0d1b2a')
plt.show()
